In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


# Importing Libraries

In [ ]:
# This will install ultralytics with compatible dependencies
!pip install torch==2.0.1 torchvision==0.15.2
!pip install ultralytics==8.0.196
!wget https://github.com/ultralytics/assets/releases/download/v0.0.0/yolov8m.pt -P /content/

ERROR: Could not find a version that satisfies the requirement torch==2.0.1 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1)
ERROR: No matching distribution found for torch==2.0.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 631.1/631.1 kB 44.2 MB/s eta 0:00:00
--2025-12-07 05:47:07--  https://github.com/ultralytics/assets/releases/download/v0.0.0/yolov8m.pt
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/521807533/0a8d2888-29a8-4d22-a130-b86c8174b33b?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-12-07T06%3A37%3A55Z&rscd=attachment%3B+filename%3Dyolov8m.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-12-07T05%3A36%3A58Z&ske=2025-1

## Inference Plastic Video 1

In [ ]:
import os
import torch
from collections import Counter

# Monkey patch torch.load ONCE to force weights_only=False
if not hasattr(torch, '_load_patched'):
    _original_load = torch.load
    def patched_load(*args, **kwargs):
        kwargs['weights_only'] = False
        return _original_load(*args, **kwargs)
    torch.load = patched_load
    torch._load_patched = True

import cv2
from ultralytics import YOLO


def track_and_classify_realtime(
    tracking_weights_path,
    classification_weights_path,
    video_path,
    output_video_path,
    roi_top=0.10,
    roi_bottom=0.70,
    conf_thres=0.4,
    classify_conf_thres=0.1,
    save_crops_dir=None,
    filter_hands=True,
    hand_conf_threshold=0.3,
):
    """
    YOLOv8 tracking + real-time classification with majority voting and hand filtering.

    Args:
        tracking_weights_path: Path to general YOLOv8 model for tracking
        classification_weights_path: Path to your custom trained model (0=metal, 1=plastic)
        video_path: Input video path
        output_video_path: Output video path
        roi_top: Top boundary of ROI (fraction of height)
        roi_bottom: Bottom boundary of ROI (fraction of height)
        conf_thres: Confidence threshold for tracking detections
        classify_conf_thres: Confidence threshold for classification model
        save_crops_dir: Directory to save cropped images
        filter_hands: Whether to filter out hand detections
        hand_conf_threshold: Confidence threshold for hand detection
    """

    # Load models
    print("Loading tracking model...")
    tracking_model = YOLO(tracking_weights_path)

    print("Loading classification model...")
    classification_model = YOLO(classification_weights_path)
    print(f"✓ Classification model loaded (Task: {classification_model.task})")

    # Load pose model for hand detection if filtering is enabled
    hand_model = None
    if filter_hands:
        print("Loading pose model for hand detection...")
        hand_model = YOLO("yolov8n-pose.pt")  # Lightweight pose model

    # Create output dirs
    if output_video_path:
        os.makedirs(os.path.dirname(output_video_path), exist_ok=True)
    if save_crops_dir is not None:
        os.makedirs(save_crops_dir, exist_ok=True)

    # Read video metadata
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    writer = cv2.VideoWriter(
        output_video_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )

    # Precompute ROI vertical boundaries
    y_min = int(roi_top * height)
    y_max = int(roi_bottom * height)

    # Storage for classifications
    all_classifications = []  # List to store all classification results
    track_classifications = {}  # Dict to store classifications per track_id

    # Tracking generator
    results_gen = tracking_model.track(
        source=video_path,
        tracker="bytetrack.yaml",
        conf=conf_thres,
        iou=0.5,
        stream=True,
        verbose=False,
    )

    frame_idx = 0

    print("\n🎬 Starting tracking and classification...")
    if filter_hands:
        print("✋ Hand filtering enabled - detections overlapping with hands will be excluded")

    for result in results_gen:
        frame = result.orig_img.copy()
        h, w = frame.shape[:2]

        # Recalculate ROI if frame shape changes
        y_min = int(roi_top * h)
        y_max = int(roi_bottom * h)

        # Detect hands in the frame if filtering is enabled
        hand_regions = []
        if filter_hands and hand_model is not None:
            hand_results = hand_model(frame, verbose=False)

            if len(hand_results) > 0 and hand_results[0].keypoints is not None:
                keypoints = hand_results[0].keypoints

                # Extract hand keypoints (wrists and hand keypoints)
                # YOLOv8-pose has 17 keypoints: 9=left_wrist, 10=right_wrist
                for person_kpts in keypoints.data:
                    if len(person_kpts) > 0:
                        # Get wrist keypoints (indices 9 and 10)
                        left_wrist = person_kpts[9][:2].cpu().numpy()  # x, y
                        right_wrist = person_kpts[10][:2].cpu().numpy()  # x, y

                        # Create expanded regions around wrists (approximate hand size)
                        hand_size = 100  # pixels radius around wrist

                        for wrist in [left_wrist, right_wrist]:
                            if wrist[0] > 0 and wrist[1] > 0:  # Valid keypoint
                                x1 = int(max(0, wrist[0] - hand_size))
                                y1 = int(max(0, wrist[1] - hand_size))
                                x2 = int(min(w, wrist[0] + hand_size))
                                y2 = int(min(h, wrist[1] + hand_size))
                                hand_regions.append((x1, y1, x2, y2))

                                # Draw hand region for visualization (optional)
                                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 1)
                                cv2.putText(frame, "HAND", (x1, y1-5),
                                          cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)

        # Draw ROI
        cv2.rectangle(frame, (0, y_min), (w, y_max), (0, 255, 255), 2)

        boxes = result.boxes

        if boxes is not None and len(boxes) > 0:
            xyxy = boxes.xyxy.cpu().numpy()
            confs = boxes.conf.cpu().numpy()
            ids = (
                boxes.id.cpu().numpy()
                if boxes.id is not None
                else [-1] * len(xyxy)
            )
            clss = boxes.cls.cpu().numpy()

            for box, score, track_id, cls_id in zip(xyxy, confs, ids, clss):
                if score < conf_thres:
                    continue

                # Skip person class detections (COCO class 0)
                if int(cls_id) == 0:
                    continue

                x1, y1, x2, y2 = box.astype(int)
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2

                # Reject detections outside ROI
                if not (y_min <= cy <= y_max):
                    continue

                # Check if detection overlaps with any hand region
                if filter_hands and hand_regions:
                    overlaps_hand = False
                    for hx1, hy1, hx2, hy2 in hand_regions:
                        # Check if bounding boxes overlap
                        if not (x2 < hx1 or x1 > hx2 or y2 < hy1 or y1 > hy2):
                            # Calculate intersection area
                            ix1 = max(x1, hx1)
                            iy1 = max(y1, hy1)
                            ix2 = min(x2, hx2)
                            iy2 = min(y2, hy2)

                            intersection_area = (ix2 - ix1) * (iy2 - iy1)
                            detection_area = (x2 - x1) * (y2 - y1)

                            # If more than 30% of detection overlaps with hand, skip it
                            if intersection_area / detection_area > 0.3:
                                overlaps_hand = True
                                break

                    if overlaps_hand:
                        # Draw rejected detection in red
                        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 1)
                        cv2.putText(frame, "HAND FILTERED", (x1, y1-5),
                                  cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)
                        continue

                # Crop the detection
                crop = frame[max(0, y1):min(h, y2), max(0, x1):min(w, x2)]

                if crop.size > 0:
                    # Default values
                    class_name = "UNKNOWN"
                    box_color = (0, 255, 0)
                    confidence = 0.0

                    # Classify the crop using custom model
                    classify_results = classification_model(crop, verbose=False, conf=classify_conf_thres)

                    if len(classify_results) > 0:
                        classify_result = classify_results[0]

                        # Check if it's a classification model
                        if hasattr(classify_result, 'probs') and classify_result.probs is not None:
                            probs = classify_result.probs.data.cpu().numpy()
                            predicted_class = int(probs.argmax())
                            confidence = float(probs.max())

                            class_name = "METAL" if predicted_class == 0 else "PLASTIC"
                            box_color = (255, 0, 0) if predicted_class == 0 else (0, 255, 255)

                            # Store classification
                            all_classifications.append(predicted_class)
                            if track_id != -1:
                                if track_id not in track_classifications:
                                    track_classifications[track_id] = []
                                track_classifications[track_id].append(predicted_class)

                        # Check if it's a detection model
                        elif hasattr(classify_result, 'boxes') and classify_result.boxes is not None:
                            classify_boxes = classify_result.boxes

                            if len(classify_boxes) > 0:
                                # Take highest confidence detection
                                best_idx = classify_boxes.conf.argmax()
                                predicted_class = int(classify_boxes.cls[best_idx].cpu().numpy())
                                confidence = float(classify_boxes.conf[best_idx].cpu().numpy())

                                class_name = "METAL" if predicted_class == 0 else "PLASTIC"
                                box_color = (255, 0, 0) if predicted_class == 0 else (0, 255, 255)

                                # Store classification
                                all_classifications.append(predicted_class)
                                if track_id != -1:
                                    if track_id not in track_classifications:
                                        track_classifications[track_id] = []
                                    track_classifications[track_id].append(predicted_class)
                            else:
                                class_name = "NO_DETECTION"

                    # Draw bounding box and label
                    label = f"ID:{int(track_id)} {class_name} {confidence:.2f}"
                    cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)
                    cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)
                    cv2.putText(
                        frame,
                        label,
                        (x1, max(0, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.5,
                        box_color,
                        2,
                        cv2.LINE_AA,
                    )

                    # Save crop with classification
                    if save_crops_dir is not None and track_id != -1 and class_name != "UNKNOWN":
                        crop_path = os.path.join(
                            save_crops_dir,
                            f"frame{frame_idx:05d}_id{int(track_id)}_{class_name}_{confidence:.3f}.jpg",
                        )
                        cv2.imwrite(crop_path, crop)

        # Add overall statistics overlay to video
        if all_classifications:
            class_counts = Counter(all_classifications)
            metal_count = class_counts.get(0, 0)
            plastic_count = class_counts.get(1, 0)
            total = len(all_classifications)

            # Determine current majority
            current_majority = "METAL" if metal_count > plastic_count else "PLASTIC"
            majority_color = (255, 0, 0) if metal_count > plastic_count else (0, 255, 255)

            # Create semi-transparent overlay for statistics
            overlay = frame.copy()
            cv2.rectangle(overlay, (10, 10), (400, 120), (0, 0, 0), -1)
            cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

            # Add text
            cv2.putText(frame, f"CURRENT VERDICT: {current_majority}", (20, 35),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, majority_color, 2, cv2.LINE_AA)
            cv2.putText(frame, f"Metal: {metal_count} ({metal_count/total*100:.1f}%)", (20, 65),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f"Plastic: {plastic_count} ({plastic_count/total*100:.1f}%)", (20, 95),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2, cv2.LINE_AA)

        writer.write(frame)
        frame_idx += 1

        if frame_idx % 30 == 0:
            print(f"Processed {frame_idx} frames...")

    writer.release()

    # Calculate final results
    print("\n" + "="*60)
    print("📊 CLASSIFICATION RESULTS")
    print("="*60)

    final_class_name = None
    metal_count = 0
    plastic_count = 0
    total = 0

    if all_classifications:
        class_counts = Counter(all_classifications)
        total = len(all_classifications)
        metal_count = class_counts.get(0, 0)
        plastic_count = class_counts.get(1, 0)

        metal_percent = (metal_count / total) * 100
        plastic_percent = (plastic_count / total) * 100

        final_class = 0 if metal_count > plastic_count else 1
        final_class_name = "METAL" if final_class == 0 else "PLASTIC"

        print(f"\n🎯 FINAL VERDICT: {final_class_name}")
        print(f"\nTotal classifications: {total}")
        print(f"  • METAL:   {metal_count:4d} ({metal_percent:5.1f}%)")
        print(f"  • PLASTIC: {plastic_count:4d} ({plastic_percent:5.1f}%)")

        # Per-track results
        print(f"\n📦 Per-Track Classifications:")
        for track_id, classifications in sorted(track_classifications.items()):
            track_counts = Counter(classifications)
            track_total = len(classifications)
            track_metal = track_counts.get(0, 0)
            track_plastic = track_counts.get(1, 0)
            track_final = 0 if track_metal > track_plastic else 1
            track_final_name = "METAL" if track_final == 0 else "PLASTIC"

            print(f"  Track ID {int(track_id):3d}: {track_final_name:7s} "
                  f"(M:{track_metal:3d}, P:{track_plastic:3d}, Total:{track_total:3d})")
    else:
        print("\n⚠️  No classifications made!")

    print("\n" + "="*60)
    print(f"✅ Saved tracked video to: {output_video_path}")
    if save_crops_dir:
        print(f"✅ Saved crops to: {save_crops_dir}")
    print("="*60)

    return {
        'final_class': final_class_name,
        'metal_count': metal_count,
        'plastic_count': plastic_count,
        'total_count': total,
        'track_results': track_classifications
    }


# --------------------------------------------------------------------
# ---------------------- CONFIGURATION -------------------------------
# --------------------------------------------------------------------

# Path to ORIGINAL YOLOv8 model for tracking (general object detection)
tracking_weights = "/content/yolov8m.pt"

# Path to YOUR CUSTOM trained model for classification (metal=0, plastic=1)
classification_weights = "/content/drive/MyDrive/Garbagedata/Garbagedata_Unzipped_Files/Ishita_Model/Garbagedata/Models_V3m_Runs/Model_V3m_Run1/weights/best.pt"

# Video and output paths
video_path = "/content/drive/MyDrive/Garbagedata/Trash Videos/IMG_2472.mov"
output_video_path = "/content/drive/MyDrive/Garbagedata/Trash Videos/outputs_trackclass/IMG_2472_op125_c1.mp4"
save_crops_dir = "/content/drive/MyDrive/Garbagedata/Trash Videos/outputs_trackclass/crops_classified/"

# ROI + confidence
roi_top = 0.10
roi_bottom = 0.70
conf = 0.4
classify_conf = 0.1  # Lower threshold for classification to catch more detections

# Run the script
results = track_and_classify_realtime(
    tracking_weights_path=tracking_weights,
    classification_weights_path=classification_weights,
    video_path=video_path,
    output_video_path=output_video_path,
    roi_top=roi_top,
    roi_bottom=roi_bottom,
    conf_thres=conf,
    classify_conf_thres=classify_conf,
    save_crops_dir=save_crops_dir,
    filter_hands=True,
    hand_conf_threshold=0.3,
)

print(f"\n🏆 FINAL RESULT: {results['final_class']}")
print(f"   Metal: {results['metal_count']} | Plastic: {results['plastic_count']}")

Loading tracking model...
Loading classification model...
✓ Classification model loaded (Task: detect)
Loading pose model for hand detection...

🎬 Starting tracking and classification...
✋ Hand filtering enabled - detections overlapping with hands will be excluded
Processed 30 frames...
Processed 60 frames...
Processed 90 frames...
Processed 120 frames...
Processed 150 frames...

📊 CLASSIFICATION RESULTS

🎯 FINAL VERDICT: PLASTIC

Total classifications: 70
  • METAL:      5 (  7.1%)
  • PLASTIC:   65 ( 92.9%)

📦 Per-Track Classifications:
  Track ID   1: PLASTIC (M:  5, P: 62, Total: 67)

✅ Saved tracked video to: /content/drive/MyDrive/Garbagedata/Trash Videos/outputs_trackclass/IMG_2472_op125_c1.mp4
✅ Saved crops to: /content/drive/MyDrive/Garbagedata/Trash Videos/outputs_trackclass/crops_classified/

🏆 FINAL RESULT: PLASTIC
   Metal: 5 | Plastic: 65


## Inference Plastic Video 2

In [ ]:
import os
import torch
from collections import Counter

# Monkey patch torch.load ONCE to force weights_only=False
if not hasattr(torch, '_load_patched'):
    _original_load = torch.load
    def patched_load(*args, **kwargs):
        kwargs['weights_only'] = False
        return _original_load(*args, **kwargs)
    torch.load = patched_load
    torch._load_patched = True

import cv2
from ultralytics import YOLO


def track_and_classify_realtime(
    tracking_weights_path,
    classification_weights_path,
    video_path,
    output_video_path,
    roi_top=0.10,
    roi_bottom=0.70,
    conf_thres=0.4,
    classify_conf_thres=0.1,
    save_crops_dir=None,
    filter_hands=True,
    hand_conf_threshold=0.3,
):
    """
    YOLOv8 tracking + real-time classification with majority voting and hand filtering.

    Args:
        tracking_weights_path: Path to general YOLOv8 model for tracking
        classification_weights_path: Path to your custom trained model (0=metal, 1=plastic)
        video_path: Input video path
        output_video_path: Output video path
        roi_top: Top boundary of ROI (fraction of height)
        roi_bottom: Bottom boundary of ROI (fraction of height)
        conf_thres: Confidence threshold for tracking detections
        classify_conf_thres: Confidence threshold for classification model
        save_crops_dir: Directory to save cropped images
        filter_hands: Whether to filter out hand detections
        hand_conf_threshold: Confidence threshold for hand detection
    """

    # Load models
    print("Loading tracking model...")
    tracking_model = YOLO(tracking_weights_path)

    print("Loading classification model...")
    classification_model = YOLO(classification_weights_path)
    print(f"✓ Classification model loaded (Task: {classification_model.task})")

    # Load pose model for hand detection if filtering is enabled
    hand_model = None
    if filter_hands:
        print("Loading pose model for hand detection...")
        hand_model = YOLO("yolov8n-pose.pt")  # Lightweight pose model

    # Create output dirs
    if output_video_path:
        os.makedirs(os.path.dirname(output_video_path), exist_ok=True)
    if save_crops_dir is not None:
        os.makedirs(save_crops_dir, exist_ok=True)

    # Read video metadata
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    writer = cv2.VideoWriter(
        output_video_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )

    # Precompute ROI vertical boundaries
    y_min = int(roi_top * height)
    y_max = int(roi_bottom * height)

    # Storage for classifications
    all_classifications = []  # List to store all classification results
    track_classifications = {}  # Dict to store classifications per track_id

    # Tracking generator
    results_gen = tracking_model.track(
        source=video_path,
        tracker="bytetrack.yaml",
        conf=conf_thres,
        iou=0.5,
        stream=True,
        verbose=False,
    )

    frame_idx = 0

    print("\n🎬 Starting tracking and classification...")
    if filter_hands:
        print("✋ Hand filtering enabled - detections overlapping with hands will be excluded")

    for result in results_gen:
        frame = result.orig_img.copy()
        h, w = frame.shape[:2]

        # Recalculate ROI if frame shape changes
        y_min = int(roi_top * h)
        y_max = int(roi_bottom * h)

        # Detect hands in the frame if filtering is enabled
        hand_regions = []
        if filter_hands and hand_model is not None:
            hand_results = hand_model(frame, verbose=False)

            if len(hand_results) > 0 and hand_results[0].keypoints is not None:
                keypoints = hand_results[0].keypoints

                # Extract hand keypoints (wrists and hand keypoints)
                # YOLOv8-pose has 17 keypoints: 9=left_wrist, 10=right_wrist
                for person_kpts in keypoints.data:
                    if len(person_kpts) > 0:
                        # Get wrist keypoints (indices 9 and 10)
                        left_wrist = person_kpts[9][:2].cpu().numpy()  # x, y
                        right_wrist = person_kpts[10][:2].cpu().numpy()  # x, y

                        # Create expanded regions around wrists (approximate hand size)
                        hand_size = 100  # pixels radius around wrist

                        for wrist in [left_wrist, right_wrist]:
                            if wrist[0] > 0 and wrist[1] > 0:  # Valid keypoint
                                x1 = int(max(0, wrist[0] - hand_size))
                                y1 = int(max(0, wrist[1] - hand_size))
                                x2 = int(min(w, wrist[0] + hand_size))
                                y2 = int(min(h, wrist[1] + hand_size))
                                hand_regions.append((x1, y1, x2, y2))

                                # Draw hand region for visualization (optional)
                                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 1)
                                cv2.putText(frame, "HAND", (x1, y1-5),
                                          cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)

        # Draw ROI
        cv2.rectangle(frame, (0, y_min), (w, y_max), (0, 255, 255), 2)

        boxes = result.boxes

        if boxes is not None and len(boxes) > 0:
            xyxy = boxes.xyxy.cpu().numpy()
            confs = boxes.conf.cpu().numpy()
            ids = (
                boxes.id.cpu().numpy()
                if boxes.id is not None
                else [-1] * len(xyxy)
            )
            clss = boxes.cls.cpu().numpy()

            for box, score, track_id, cls_id in zip(xyxy, confs, ids, clss):
                if score < conf_thres:
                    continue

                # Skip person class detections (COCO class 0)
                if int(cls_id) == 0:
                    continue

                x1, y1, x2, y2 = box.astype(int)
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2

                # Reject detections outside ROI
                if not (y_min <= cy <= y_max):
                    continue

                # Check if detection overlaps with any hand region
                if filter_hands and hand_regions:
                    overlaps_hand = False
                    for hx1, hy1, hx2, hy2 in hand_regions:
                        # Check if bounding boxes overlap
                        if not (x2 < hx1 or x1 > hx2 or y2 < hy1 or y1 > hy2):
                            # Calculate intersection area
                            ix1 = max(x1, hx1)
                            iy1 = max(y1, hy1)
                            ix2 = min(x2, hx2)
                            iy2 = min(y2, hy2)

                            intersection_area = (ix2 - ix1) * (iy2 - iy1)
                            detection_area = (x2 - x1) * (y2 - y1)

                            # If more than 30% of detection overlaps with hand, skip it
                            if intersection_area / detection_area > 0.3:
                                overlaps_hand = True
                                break

                    if overlaps_hand:
                        # Draw rejected detection in red
                        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 1)
                        cv2.putText(frame, "HAND FILTERED", (x1, y1-5),
                                  cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)
                        continue

                # Crop the detection
                crop = frame[max(0, y1):min(h, y2), max(0, x1):min(w, x2)]

                if crop.size > 0:
                    # Default values
                    class_name = "UNKNOWN"
                    box_color = (0, 255, 0)
                    confidence = 0.0

                    # Classify the crop using custom model
                    classify_results = classification_model(crop, verbose=False, conf=classify_conf_thres)

                    if len(classify_results) > 0:
                        classify_result = classify_results[0]

                        # Check if it's a classification model
                        if hasattr(classify_result, 'probs') and classify_result.probs is not None:
                            probs = classify_result.probs.data.cpu().numpy()
                            predicted_class = int(probs.argmax())
                            confidence = float(probs.max())

                            class_name = "METAL" if predicted_class == 0 else "PLASTIC"
                            box_color = (255, 0, 0) if predicted_class == 0 else (0, 255, 255)

                            # Store classification
                            all_classifications.append(predicted_class)
                            if track_id != -1:
                                if track_id not in track_classifications:
                                    track_classifications[track_id] = []
                                track_classifications[track_id].append(predicted_class)

                        # Check if it's a detection model
                        elif hasattr(classify_result, 'boxes') and classify_result.boxes is not None:
                            classify_boxes = classify_result.boxes

                            if len(classify_boxes) > 0:
                                # Take highest confidence detection
                                best_idx = classify_boxes.conf.argmax()
                                predicted_class = int(classify_boxes.cls[best_idx].cpu().numpy())
                                confidence = float(classify_boxes.conf[best_idx].cpu().numpy())

                                class_name = "METAL" if predicted_class == 0 else "PLASTIC"
                                box_color = (255, 0, 0) if predicted_class == 0 else (0, 255, 255)

                                # Store classification
                                all_classifications.append(predicted_class)
                                if track_id != -1:
                                    if track_id not in track_classifications:
                                        track_classifications[track_id] = []
                                    track_classifications[track_id].append(predicted_class)
                            else:
                                class_name = "NO_DETECTION"

                    # Draw bounding box and label
                    label = f"ID:{int(track_id)} {class_name} {confidence:.2f}"
                    cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)
                    cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)
                    cv2.putText(
                        frame,
                        label,
                        (x1, max(0, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.5,
                        box_color,
                        2,
                        cv2.LINE_AA,
                    )

                    # Save crop with classification
                    if save_crops_dir is not None and track_id != -1 and class_name != "UNKNOWN":
                        crop_path = os.path.join(
                            save_crops_dir,
                            f"frame{frame_idx:05d}_id{int(track_id)}_{class_name}_{confidence:.3f}.jpg",
                        )
                        cv2.imwrite(crop_path, crop)

        # Add overall statistics overlay to video
        if all_classifications:
            class_counts = Counter(all_classifications)
            metal_count = class_counts.get(0, 0)
            plastic_count = class_counts.get(1, 0)
            total = len(all_classifications)

            # Determine current majority
            current_majority = "METAL" if metal_count > plastic_count else "PLASTIC"
            majority_color = (255, 0, 0) if metal_count > plastic_count else (0, 255, 255)

            # Create semi-transparent overlay for statistics
            overlay = frame.copy()
            cv2.rectangle(overlay, (10, 10), (400, 120), (0, 0, 0), -1)
            cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

            # Add text
            cv2.putText(frame, f"CURRENT VERDICT: {current_majority}", (20, 35),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, majority_color, 2, cv2.LINE_AA)
            cv2.putText(frame, f"Metal: {metal_count} ({metal_count/total*100:.1f}%)", (20, 65),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f"Plastic: {plastic_count} ({plastic_count/total*100:.1f}%)", (20, 95),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2, cv2.LINE_AA)

        writer.write(frame)
        frame_idx += 1

        if frame_idx % 30 == 0:
            print(f"Processed {frame_idx} frames...")

    writer.release()

    # Calculate final results
    print("\n" + "="*60)
    print("📊 CLASSIFICATION RESULTS")
    print("="*60)

    final_class_name = None
    metal_count = 0
    plastic_count = 0
    total = 0

    if all_classifications:
        class_counts = Counter(all_classifications)
        total = len(all_classifications)
        metal_count = class_counts.get(0, 0)
        plastic_count = class_counts.get(1, 0)

        metal_percent = (metal_count / total) * 100
        plastic_percent = (plastic_count / total) * 100

        final_class = 0 if metal_count > plastic_count else 1
        final_class_name = "METAL" if final_class == 0 else "PLASTIC"

        print(f"\n🎯 FINAL VERDICT: {final_class_name}")
        print(f"\nTotal classifications: {total}")
        print(f"  • METAL:   {metal_count:4d} ({metal_percent:5.1f}%)")
        print(f"  • PLASTIC: {plastic_count:4d} ({plastic_percent:5.1f}%)")

        # Per-track results
        print(f"\n📦 Per-Track Classifications:")
        for track_id, classifications in sorted(track_classifications.items()):
            track_counts = Counter(classifications)
            track_total = len(classifications)
            track_metal = track_counts.get(0, 0)
            track_plastic = track_counts.get(1, 0)
            track_final = 0 if track_metal > track_plastic else 1
            track_final_name = "METAL" if track_final == 0 else "PLASTIC"

            print(f"  Track ID {int(track_id):3d}: {track_final_name:7s} "
                  f"(M:{track_metal:3d}, P:{track_plastic:3d}, Total:{track_total:3d})")
    else:
        print("\n⚠️  No classifications made!")

    print("\n" + "="*60)
    print(f"✅ Saved tracked video to: {output_video_path}")
    if save_crops_dir:
        print(f"✅ Saved crops to: {save_crops_dir}")
    print("="*60)

    return {
        'final_class': final_class_name,
        'metal_count': metal_count,
        'plastic_count': plastic_count,
        'total_count': total,
        'track_results': track_classifications
    }


# --------------------------------------------------------------------
# ---------------------- CONFIGURATION -------------------------------
# --------------------------------------------------------------------

# Path to ORIGINAL YOLOv8 model for tracking (general object detection)
tracking_weights = "/content/yolov8m.pt"

# Path to YOUR CUSTOM trained model for classification (metal=0, plastic=1)
classification_weights = "/content/drive/MyDrive/Garbagedata/Garbagedata_Unzipped_Files/Ishita_Model/Garbagedata/Models_V3m_Runs/Model_V3m_Run1/weights/best.pt"

# Video and output paths
video_path = "/content/drive/MyDrive/Garbagedata/Trash Videos/IMG_2473.mov"
output_video_path = "/content/drive/MyDrive/Garbagedata/Trash Videos/outputs_trackclass/IMG_2473_op125_c1.mp4"
save_crops_dir = "/content/drive/MyDrive/Garbagedata/Trash Videos/outputs_trackclass/crops_classified/"

# ROI + confidence
roi_top = 0.10
roi_bottom = 0.70
conf = 0.4
classify_conf = 0.1  # Lower threshold for classification to catch more detections

# Run the script
results = track_and_classify_realtime(
    tracking_weights_path=tracking_weights,
    classification_weights_path=classification_weights,
    video_path=video_path,
    output_video_path=output_video_path,
    roi_top=roi_top,
    roi_bottom=roi_bottom,
    conf_thres=conf,
    classify_conf_thres=classify_conf,
    save_crops_dir=save_crops_dir,
    filter_hands=True,
    hand_conf_threshold=0.3,
)

print(f"\n🏆 FINAL RESULT: {results['final_class']}")
print(f"   Metal: {results['metal_count']} | Plastic: {results['plastic_count']}")

Loading tracking model...
Loading classification model...
✓ Classification model loaded (Task: detect)
Loading pose model for hand detection...

🎬 Starting tracking and classification...
✋ Hand filtering enabled - detections overlapping with hands will be excluded
Processed 30 frames...
Processed 60 frames...
Processed 90 frames...
Processed 120 frames...
Processed 150 frames...

📊 CLASSIFICATION RESULTS

🎯 FINAL VERDICT: PLASTIC

Total classifications: 1
  • METAL:      0 (  0.0%)
  • PLASTIC:    1 (100.0%)

📦 Per-Track Classifications:

✅ Saved tracked video to: /content/drive/MyDrive/Garbagedata/Trash Videos/outputs_trackclass/IMG_2473_op125_c1.mp4
✅ Saved crops to: /content/drive/MyDrive/Garbagedata/Trash Videos/outputs_trackclass/crops_classified/

🏆 FINAL RESULT: PLASTIC
   Metal: 0 | Plastic: 1


## Inference Metal Video 1

In [ ]:
import os
import torch
from collections import Counter

# Monkey patch torch.load ONCE to force weights_only=False
if not hasattr(torch, '_load_patched'):
    _original_load = torch.load
    def patched_load(*args, **kwargs):
        kwargs['weights_only'] = False
        return _original_load(*args, **kwargs)
    torch.load = patched_load
    torch._load_patched = True

import cv2
from ultralytics import YOLO


def track_and_classify_realtime(
    tracking_weights_path,
    classification_weights_path,
    video_path,
    output_video_path,
    roi_top=0.10,
    roi_bottom=0.70,
    conf_thres=0.4,
    classify_conf_thres=0.1,
    save_crops_dir=None,
    filter_hands=True,
    hand_conf_threshold=0.3,
):
    """
    YOLOv8 tracking + real-time classification with majority voting and hand filtering.

    Args:
        tracking_weights_path: Path to general YOLOv8 model for tracking
        classification_weights_path: Path to your custom trained model (0=metal, 1=plastic)
        video_path: Input video path
        output_video_path: Output video path
        roi_top: Top boundary of ROI (fraction of height)
        roi_bottom: Bottom boundary of ROI (fraction of height)
        conf_thres: Confidence threshold for tracking detections
        classify_conf_thres: Confidence threshold for classification model
        save_crops_dir: Directory to save cropped images
        filter_hands: Whether to filter out hand detections
        hand_conf_threshold: Confidence threshold for hand detection
    """

    # Load models
    print("Loading tracking model...")
    tracking_model = YOLO(tracking_weights_path)

    print("Loading classification model...")
    classification_model = YOLO(classification_weights_path)
    print(f"✓ Classification model loaded (Task: {classification_model.task})")

    # Load pose model for hand detection if filtering is enabled
    hand_model = None
    if filter_hands:
        print("Loading pose model for hand detection...")
        hand_model = YOLO("yolov8n-pose.pt")  # Lightweight pose model

    # Create output dirs
    if output_video_path:
        os.makedirs(os.path.dirname(output_video_path), exist_ok=True)
    if save_crops_dir is not None:
        os.makedirs(save_crops_dir, exist_ok=True)

    # Read video metadata
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    writer = cv2.VideoWriter(
        output_video_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )

    # Precompute ROI vertical boundaries
    y_min = int(roi_top * height)
    y_max = int(roi_bottom * height)

    # Storage for classifications
    all_classifications = []  # List to store all classification results
    track_classifications = {}  # Dict to store classifications per track_id

    # Tracking generator
    results_gen = tracking_model.track(
        source=video_path,
        tracker="bytetrack.yaml",
        conf=conf_thres,
        iou=0.5,
        stream=True,
        verbose=False,
    )

    frame_idx = 0

    print("\n🎬 Starting tracking and classification...")
    if filter_hands:
        print("✋ Hand filtering enabled - detections overlapping with hands will be excluded")

    for result in results_gen:
        frame = result.orig_img.copy()
        h, w = frame.shape[:2]

        # Recalculate ROI if frame shape changes
        y_min = int(roi_top * h)
        y_max = int(roi_bottom * h)

        # Detect hands in the frame if filtering is enabled
        hand_regions = []
        if filter_hands and hand_model is not None:
            hand_results = hand_model(frame, verbose=False)

            if len(hand_results) > 0 and hand_results[0].keypoints is not None:
                keypoints = hand_results[0].keypoints

                # Extract hand keypoints (wrists and hand keypoints)
                # YOLOv8-pose has 17 keypoints: 9=left_wrist, 10=right_wrist
                for person_kpts in keypoints.data:
                    if len(person_kpts) > 0:
                        # Get wrist keypoints (indices 9 and 10)
                        left_wrist = person_kpts[9][:2].cpu().numpy()  # x, y
                        right_wrist = person_kpts[10][:2].cpu().numpy()  # x, y

                        # Create expanded regions around wrists (approximate hand size)
                        hand_size = 100  # pixels radius around wrist

                        for wrist in [left_wrist, right_wrist]:
                            if wrist[0] > 0 and wrist[1] > 0:  # Valid keypoint
                                x1 = int(max(0, wrist[0] - hand_size))
                                y1 = int(max(0, wrist[1] - hand_size))
                                x2 = int(min(w, wrist[0] + hand_size))
                                y2 = int(min(h, wrist[1] + hand_size))
                                hand_regions.append((x1, y1, x2, y2))

                                # Draw hand region for visualization (optional)
                                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 1)
                                cv2.putText(frame, "HAND", (x1, y1-5),
                                          cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)

        # Draw ROI
        cv2.rectangle(frame, (0, y_min), (w, y_max), (0, 255, 255), 2)

        boxes = result.boxes

        if boxes is not None and len(boxes) > 0:
            xyxy = boxes.xyxy.cpu().numpy()
            confs = boxes.conf.cpu().numpy()
            ids = (
                boxes.id.cpu().numpy()
                if boxes.id is not None
                else [-1] * len(xyxy)
            )
            clss = boxes.cls.cpu().numpy()

            for box, score, track_id, cls_id in zip(xyxy, confs, ids, clss):
                if score < conf_thres:
                    continue

                # Skip person class detections (COCO class 0)
                if int(cls_id) == 0:
                    continue

                x1, y1, x2, y2 = box.astype(int)
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2

                # Reject detections outside ROI
                if not (y_min <= cy <= y_max):
                    continue

                # Check if detection overlaps with any hand region
                if filter_hands and hand_regions:
                    overlaps_hand = False
                    for hx1, hy1, hx2, hy2 in hand_regions:
                        # Check if bounding boxes overlap
                        if not (x2 < hx1 or x1 > hx2 or y2 < hy1 or y1 > hy2):
                            # Calculate intersection area
                            ix1 = max(x1, hx1)
                            iy1 = max(y1, hy1)
                            ix2 = min(x2, hx2)
                            iy2 = min(y2, hy2)

                            intersection_area = (ix2 - ix1) * (iy2 - iy1)
                            detection_area = (x2 - x1) * (y2 - y1)

                            # If more than 30% of detection overlaps with hand, skip it
                            if intersection_area / detection_area > 0.3:
                                overlaps_hand = True
                                break

                    if overlaps_hand:
                        # Draw rejected detection in red
                        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 1)
                        cv2.putText(frame, "HAND FILTERED", (x1, y1-5),
                                  cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)
                        continue

                # Crop the detection
                crop = frame[max(0, y1):min(h, y2), max(0, x1):min(w, x2)]

                if crop.size > 0:
                    # Default values
                    class_name = "UNKNOWN"
                    box_color = (0, 255, 0)
                    confidence = 0.0

                    # Classify the crop using custom model
                    classify_results = classification_model(crop, verbose=False, conf=classify_conf_thres)

                    if len(classify_results) > 0:
                        classify_result = classify_results[0]

                        # Check if it's a classification model
                        if hasattr(classify_result, 'probs') and classify_result.probs is not None:
                            probs = classify_result.probs.data.cpu().numpy()
                            predicted_class = int(probs.argmax())
                            confidence = float(probs.max())

                            class_name = "METAL" if predicted_class == 0 else "PLASTIC"
                            box_color = (255, 0, 0) if predicted_class == 0 else (0, 255, 255)

                            # Store classification
                            all_classifications.append(predicted_class)
                            if track_id != -1:
                                if track_id not in track_classifications:
                                    track_classifications[track_id] = []
                                track_classifications[track_id].append(predicted_class)

                        # Check if it's a detection model
                        elif hasattr(classify_result, 'boxes') and classify_result.boxes is not None:
                            classify_boxes = classify_result.boxes

                            if len(classify_boxes) > 0:
                                # Take highest confidence detection
                                best_idx = classify_boxes.conf.argmax()
                                predicted_class = int(classify_boxes.cls[best_idx].cpu().numpy())
                                confidence = float(classify_boxes.conf[best_idx].cpu().numpy())

                                class_name = "METAL" if predicted_class == 0 else "PLASTIC"
                                box_color = (255, 0, 0) if predicted_class == 0 else (0, 255, 255)

                                # Store classification
                                all_classifications.append(predicted_class)
                                if track_id != -1:
                                    if track_id not in track_classifications:
                                        track_classifications[track_id] = []
                                    track_classifications[track_id].append(predicted_class)
                            else:
                                class_name = "NO_DETECTION"

                    # Draw bounding box and label
                    label = f"ID:{int(track_id)} {class_name} {confidence:.2f}"
                    cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)
                    cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)
                    cv2.putText(
                        frame,
                        label,
                        (x1, max(0, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.5,
                        box_color,
                        2,
                        cv2.LINE_AA,
                    )

                    # Save crop with classification
                    if save_crops_dir is not None and track_id != -1 and class_name != "UNKNOWN":
                        crop_path = os.path.join(
                            save_crops_dir,
                            f"frame{frame_idx:05d}_id{int(track_id)}_{class_name}_{confidence:.3f}.jpg",
                        )
                        cv2.imwrite(crop_path, crop)

        # Add overall statistics overlay to video
        if all_classifications:
            class_counts = Counter(all_classifications)
            metal_count = class_counts.get(0, 0)
            plastic_count = class_counts.get(1, 0)
            total = len(all_classifications)

            # Determine current majority
            current_majority = "METAL" if metal_count > plastic_count else "PLASTIC"
            majority_color = (255, 0, 0) if metal_count > plastic_count else (0, 255, 255)

            # Create semi-transparent overlay for statistics
            overlay = frame.copy()
            cv2.rectangle(overlay, (10, 10), (400, 120), (0, 0, 0), -1)
            cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

            # Add text
            cv2.putText(frame, f"CURRENT VERDICT: {current_majority}", (20, 35),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, majority_color, 2, cv2.LINE_AA)
            cv2.putText(frame, f"Metal: {metal_count} ({metal_count/total*100:.1f}%)", (20, 65),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f"Plastic: {plastic_count} ({plastic_count/total*100:.1f}%)", (20, 95),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2, cv2.LINE_AA)

        writer.write(frame)
        frame_idx += 1

        if frame_idx % 30 == 0:
            print(f"Processed {frame_idx} frames...")

    writer.release()

    # Calculate final results
    print("\n" + "="*60)
    print("📊 CLASSIFICATION RESULTS")
    print("="*60)

    final_class_name = None
    metal_count = 0
    plastic_count = 0
    total = 0

    if all_classifications:
        class_counts = Counter(all_classifications)
        total = len(all_classifications)
        metal_count = class_counts.get(0, 0)
        plastic_count = class_counts.get(1, 0)

        metal_percent = (metal_count / total) * 100
        plastic_percent = (plastic_count / total) * 100

        final_class = 0 if metal_count > plastic_count else 1
        final_class_name = "METAL" if final_class == 0 else "PLASTIC"

        print(f"\n🎯 FINAL VERDICT: {final_class_name}")
        print(f"\nTotal classifications: {total}")
        print(f"  • METAL:   {metal_count:4d} ({metal_percent:5.1f}%)")
        print(f"  • PLASTIC: {plastic_count:4d} ({plastic_percent:5.1f}%)")

        # Per-track results
        print(f"\n📦 Per-Track Classifications:")
        for track_id, classifications in sorted(track_classifications.items()):
            track_counts = Counter(classifications)
            track_total = len(classifications)
            track_metal = track_counts.get(0, 0)
            track_plastic = track_counts.get(1, 0)
            track_final = 0 if track_metal > track_plastic else 1
            track_final_name = "METAL" if track_final == 0 else "PLASTIC"

            print(f"  Track ID {int(track_id):3d}: {track_final_name:7s} "
                  f"(M:{track_metal:3d}, P:{track_plastic:3d}, Total:{track_total:3d})")
    else:
        print("\n⚠️  No classifications made!")

    print("\n" + "="*60)
    print(f"✅ Saved tracked video to: {output_video_path}")
    if save_crops_dir:
        print(f"✅ Saved crops to: {save_crops_dir}")
    print("="*60)

    return {
        'final_class': final_class_name,
        'metal_count': metal_count,
        'plastic_count': plastic_count,
        'total_count': total,
        'track_results': track_classifications
    }


# --------------------------------------------------------------------
# ---------------------- CONFIGURATION -------------------------------
# --------------------------------------------------------------------

# Path to ORIGINAL YOLOv8 model for tracking (general object detection)
tracking_weights = "/content/yolov8m.pt"

# Path to YOUR CUSTOM trained model for classification (metal=0, plastic=1)
classification_weights = "/content/drive/MyDrive/Garbagedata/Garbagedata_Unzipped_Files/Ishita_Model/Garbagedata/Models_V3m_Runs/Model_V3m_Run1/weights/best.pt"

# Video and output paths
video_path = "/content/drive/MyDrive/Garbagedata/Plainbackground_videos/metal/slowmetaltrashvideo8.mp4"
output_video_path = "/content/drive/MyDrive/Garbagedata/Trash Videos/outputs_trackclass/slowmetaltrashvideo8_op.mp4"
save_crops_dir = "/content/drive/MyDrive/Garbagedata/Trash Videos/outputs_trackclass/crops_classified/"

# ROI + confidence
roi_top = 0.10
roi_bottom = 0.70
conf = 0.4
classify_conf = 0.1  # Lower threshold for classification to catch more detections

# Run the script
results = track_and_classify_realtime(
    tracking_weights_path=tracking_weights,
    classification_weights_path=classification_weights,
    video_path=video_path,
    output_video_path=output_video_path,
    roi_top=roi_top,
    roi_bottom=roi_bottom,
    conf_thres=conf,
    classify_conf_thres=classify_conf,
    save_crops_dir=save_crops_dir,
    filter_hands=True,
    hand_conf_threshold=0.3,
)

print(f"\n🏆 FINAL RESULT: {results['final_class']}")
print(f"   Metal: {results['metal_count']} | Plastic: {results['plastic_count']}")

Loading tracking model...
Loading classification model...


✓ Classification model loaded (Task: detect)
Loading pose model for hand detection...


100%|██████████| 6.51M/6.51M [00:00<00:00, 93.5MB/s]
requirements: Ultralytics requirement ['lapx>=0.5.2'] not found, attempting AutoUpdate...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 52.3 MB/s eta 0:00:00

requirements: AutoUpdate success ✅ 3.0s, installed 1 package: ['lapx>=0.5.2']
requirements: ⚠️ Restart runtime or rerun command for updates to take effect




🎬 Starting tracking and classification...
✋ Hand filtering enabled - detections overlapping with hands will be excluded
Processed 30 frames...

📊 CLASSIFICATION RESULTS

🎯 FINAL VERDICT: METAL

Total classifications: 6
  • METAL:      5 ( 83.3%)
  • PLASTIC:    1 ( 16.7%)

📦 Per-Track Classifications:

✅ Saved tracked video to: /content/drive/MyDrive/Garbagedata/Trash Videos/outputs_trackclass/slowmetaltrashvideo8_op.mp4
✅ Saved crops to: /content/drive/MyDrive/Garbagedata/Trash Videos/outputs_trackclass/crops_classified/

🏆 FINAL RESULT: METAL
   Metal: 5 | Plastic: 1


## Inference Metal Video 2


In [ ]:
import os
import torch
from collections import Counter

# Monkey patch torch.load ONCE to force weights_only=False
if not hasattr(torch, '_load_patched'):
    _original_load = torch.load
    def patched_load(*args, **kwargs):
        kwargs['weights_only'] = False
        return _original_load(*args, **kwargs)
    torch.load = patched_load
    torch._load_patched = True

import cv2
from ultralytics import YOLO


def track_and_classify_realtime(
    tracking_weights_path,
    classification_weights_path,
    video_path,
    output_video_path,
    roi_top=0.10,
    roi_bottom=0.70,
    conf_thres=0.4,
    classify_conf_thres=0.1,
    save_crops_dir=None,
    filter_hands=True,
    hand_conf_threshold=0.3,
):
    """
    YOLOv8 tracking + real-time classification with majority voting and hand filtering.

    Args:
        tracking_weights_path: Path to general YOLOv8 model for tracking
        classification_weights_path: Path to your custom trained model (0=metal, 1=plastic)
        video_path: Input video path
        output_video_path: Output video path
        roi_top: Top boundary of ROI (fraction of height)
        roi_bottom: Bottom boundary of ROI (fraction of height)
        conf_thres: Confidence threshold for tracking detections
        classify_conf_thres: Confidence threshold for classification model
        save_crops_dir: Directory to save cropped images
        filter_hands: Whether to filter out hand detections
        hand_conf_threshold: Confidence threshold for hand detection
    """

    # Load models
    print("Loading tracking model...")
    tracking_model = YOLO(tracking_weights_path)

    print("Loading classification model...")
    classification_model = YOLO(classification_weights_path)
    print(f"✓ Classification model loaded (Task: {classification_model.task})")

    # Load pose model for hand detection if filtering is enabled
    hand_model = None
    if filter_hands:
        print("Loading pose model for hand detection...")
        hand_model = YOLO("yolov8n-pose.pt")  # Lightweight pose model

    # Create output dirs
    if output_video_path:
        os.makedirs(os.path.dirname(output_video_path), exist_ok=True)
    if save_crops_dir is not None:
        os.makedirs(save_crops_dir, exist_ok=True)

    # Read video metadata
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    writer = cv2.VideoWriter(
        output_video_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )

    # Precompute ROI vertical boundaries
    y_min = int(roi_top * height)
    y_max = int(roi_bottom * height)

    # Storage for classifications
    all_classifications = []  # List to store all classification results
    track_classifications = {}  # Dict to store classifications per track_id

    # Tracking generator
    results_gen = tracking_model.track(
        source=video_path,
        tracker="bytetrack.yaml",
        conf=conf_thres,
        iou=0.5,
        stream=True,
        verbose=False,
    )

    frame_idx = 0

    print("\n🎬 Starting tracking and classification...")
    if filter_hands:
        print("✋ Hand filtering enabled - detections overlapping with hands will be excluded")

    for result in results_gen:
        frame = result.orig_img.copy()
        h, w = frame.shape[:2]

        # Recalculate ROI if frame shape changes
        y_min = int(roi_top * h)
        y_max = int(roi_bottom * h)

        # Detect hands in the frame if filtering is enabled
        hand_regions = []
        if filter_hands and hand_model is not None:
            hand_results = hand_model(frame, verbose=False)

            if len(hand_results) > 0 and hand_results[0].keypoints is not None:
                keypoints = hand_results[0].keypoints

                # Extract hand keypoints (wrists and hand keypoints)
                # YOLOv8-pose has 17 keypoints: 9=left_wrist, 10=right_wrist
                for person_kpts in keypoints.data:
                    if len(person_kpts) > 0:
                        # Get wrist keypoints (indices 9 and 10)
                        left_wrist = person_kpts[9][:2].cpu().numpy()  # x, y
                        right_wrist = person_kpts[10][:2].cpu().numpy()  # x, y

                        # Create expanded regions around wrists (approximate hand size)
                        hand_size = 100  # pixels radius around wrist

                        for wrist in [left_wrist, right_wrist]:
                            if wrist[0] > 0 and wrist[1] > 0:  # Valid keypoint
                                x1 = int(max(0, wrist[0] - hand_size))
                                y1 = int(max(0, wrist[1] - hand_size))
                                x2 = int(min(w, wrist[0] + hand_size))
                                y2 = int(min(h, wrist[1] + hand_size))
                                hand_regions.append((x1, y1, x2, y2))

                                # Draw hand region for visualization (optional)
                                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 1)
                                cv2.putText(frame, "HAND", (x1, y1-5),
                                          cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)

        # Draw ROI
        cv2.rectangle(frame, (0, y_min), (w, y_max), (0, 255, 255), 2)

        boxes = result.boxes

        if boxes is not None and len(boxes) > 0:
            xyxy = boxes.xyxy.cpu().numpy()
            confs = boxes.conf.cpu().numpy()
            ids = (
                boxes.id.cpu().numpy()
                if boxes.id is not None
                else [-1] * len(xyxy)
            )
            clss = boxes.cls.cpu().numpy()

            for box, score, track_id, cls_id in zip(xyxy, confs, ids, clss):
                if score < conf_thres:
                    continue

                # Skip person class detections (COCO class 0)
                if int(cls_id) == 0:
                    continue

                x1, y1, x2, y2 = box.astype(int)
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2

                # Reject detections outside ROI
                if not (y_min <= cy <= y_max):
                    continue

                # Check if detection overlaps with any hand region
                if filter_hands and hand_regions:
                    overlaps_hand = False
                    for hx1, hy1, hx2, hy2 in hand_regions:
                        # Check if bounding boxes overlap
                        if not (x2 < hx1 or x1 > hx2 or y2 < hy1 or y1 > hy2):
                            # Calculate intersection area
                            ix1 = max(x1, hx1)
                            iy1 = max(y1, hy1)
                            ix2 = min(x2, hx2)
                            iy2 = min(y2, hy2)

                            intersection_area = (ix2 - ix1) * (iy2 - iy1)
                            detection_area = (x2 - x1) * (y2 - y1)

                            # If more than 30% of detection overlaps with hand, skip it
                            if intersection_area / detection_area > 0.3:
                                overlaps_hand = True
                                break

                    if overlaps_hand:
                        # Draw rejected detection in red
                        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 1)
                        cv2.putText(frame, "HAND FILTERED", (x1, y1-5),
                                  cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)
                        continue

                # Crop the detection
                crop = frame[max(0, y1):min(h, y2), max(0, x1):min(w, x2)]

                if crop.size > 0:
                    # Default values
                    class_name = "UNKNOWN"
                    box_color = (0, 255, 0)
                    confidence = 0.0

                    # Classify the crop using custom model
                    classify_results = classification_model(crop, verbose=False, conf=classify_conf_thres)

                    if len(classify_results) > 0:
                        classify_result = classify_results[0]

                        # Check if it's a classification model
                        if hasattr(classify_result, 'probs') and classify_result.probs is not None:
                            probs = classify_result.probs.data.cpu().numpy()
                            predicted_class = int(probs.argmax())
                            confidence = float(probs.max())

                            class_name = "METAL" if predicted_class == 0 else "PLASTIC"
                            box_color = (255, 0, 0) if predicted_class == 0 else (0, 255, 255)

                            # Store classification
                            all_classifications.append(predicted_class)
                            if track_id != -1:
                                if track_id not in track_classifications:
                                    track_classifications[track_id] = []
                                track_classifications[track_id].append(predicted_class)

                        # Check if it's a detection model
                        elif hasattr(classify_result, 'boxes') and classify_result.boxes is not None:
                            classify_boxes = classify_result.boxes

                            if len(classify_boxes) > 0:
                                # Take highest confidence detection
                                best_idx = classify_boxes.conf.argmax()
                                predicted_class = int(classify_boxes.cls[best_idx].cpu().numpy())
                                confidence = float(classify_boxes.conf[best_idx].cpu().numpy())

                                class_name = "METAL" if predicted_class == 0 else "PLASTIC"
                                box_color = (255, 0, 0) if predicted_class == 0 else (0, 255, 255)

                                # Store classification
                                all_classifications.append(predicted_class)
                                if track_id != -1:
                                    if track_id not in track_classifications:
                                        track_classifications[track_id] = []
                                    track_classifications[track_id].append(predicted_class)
                            else:
                                class_name = "NO_DETECTION"

                    # Draw bounding box and label
                    label = f"ID:{int(track_id)} {class_name} {confidence:.2f}"
                    cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)
                    cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)
                    cv2.putText(
                        frame,
                        label,
                        (x1, max(0, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.5,
                        box_color,
                        2,
                        cv2.LINE_AA,
                    )

                    # Save crop with classification
                    if save_crops_dir is not None and track_id != -1 and class_name != "UNKNOWN":
                        crop_path = os.path.join(
                            save_crops_dir,
                            f"frame{frame_idx:05d}_id{int(track_id)}_{class_name}_{confidence:.3f}.jpg",
                        )
                        cv2.imwrite(crop_path, crop)

        # Add overall statistics overlay to video
        if all_classifications:
            class_counts = Counter(all_classifications)
            metal_count = class_counts.get(0, 0)
            plastic_count = class_counts.get(1, 0)
            total = len(all_classifications)

            # Determine current majority
            current_majority = "METAL" if metal_count > plastic_count else "PLASTIC"
            majority_color = (255, 0, 0) if metal_count > plastic_count else (0, 255, 255)

            # Create semi-transparent overlay for statistics
            overlay = frame.copy()
            cv2.rectangle(overlay, (10, 10), (400, 120), (0, 0, 0), -1)
            cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

            # Add text
            cv2.putText(frame, f"CURRENT VERDICT: {current_majority}", (20, 35),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, majority_color, 2, cv2.LINE_AA)
            cv2.putText(frame, f"Metal: {metal_count} ({metal_count/total*100:.1f}%)", (20, 65),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f"Plastic: {plastic_count} ({plastic_count/total*100:.1f}%)", (20, 95),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2, cv2.LINE_AA)

        writer.write(frame)
        frame_idx += 1

        if frame_idx % 30 == 0:
            print(f"Processed {frame_idx} frames...")

    writer.release()

    # Calculate final results
    print("\n" + "="*60)
    print("📊 CLASSIFICATION RESULTS")
    print("="*60)

    final_class_name = None
    metal_count = 0
    plastic_count = 0
    total = 0

    if all_classifications:
        class_counts = Counter(all_classifications)
        total = len(all_classifications)
        metal_count = class_counts.get(0, 0)
        plastic_count = class_counts.get(1, 0)

        metal_percent = (metal_count / total) * 100
        plastic_percent = (plastic_count / total) * 100

        final_class = 0 if metal_count > plastic_count else 1
        final_class_name = "METAL" if final_class == 0 else "PLASTIC"

        print(f"\n🎯 FINAL VERDICT: {final_class_name}")
        print(f"\nTotal classifications: {total}")
        print(f"  • METAL:   {metal_count:4d} ({metal_percent:5.1f}%)")
        print(f"  • PLASTIC: {plastic_count:4d} ({plastic_percent:5.1f}%)")

        # Per-track results
        print(f"\n📦 Per-Track Classifications:")
        for track_id, classifications in sorted(track_classifications.items()):
            track_counts = Counter(classifications)
            track_total = len(classifications)
            track_metal = track_counts.get(0, 0)
            track_plastic = track_counts.get(1, 0)
            track_final = 0 if track_metal > track_plastic else 1
            track_final_name = "METAL" if track_final == 0 else "PLASTIC"

            print(f"  Track ID {int(track_id):3d}: {track_final_name:7s} "
                  f"(M:{track_metal:3d}, P:{track_plastic:3d}, Total:{track_total:3d})")
    else:
        print("\n⚠️  No classifications made!")

    print("\n" + "="*60)
    print(f"✅ Saved tracked video to: {output_video_path}")
    if save_crops_dir:
        print(f"✅ Saved crops to: {save_crops_dir}")
    print("="*60)

    return {
        'final_class': final_class_name,
        'metal_count': metal_count,
        'plastic_count': plastic_count,
        'total_count': total,
        'track_results': track_classifications
    }


# --------------------------------------------------------------------
# ---------------------- CONFIGURATION -------------------------------
# --------------------------------------------------------------------

# Path to ORIGINAL YOLOv8 model for tracking (general object detection)
tracking_weights = "/content/yolov8m.pt"

# Path to YOUR CUSTOM trained model for classification (metal=0, plastic=1)
classification_weights = "/content/drive/MyDrive/Garbagedata/Garbagedata_Unzipped_Files/Ishita_Model/Garbagedata/Models_V3m_Runs/Model_V3m_Run1/weights/best.pt"

# Video and output paths
video_path = "/content/drive/MyDrive/Garbagedata/Plainbackground_videos/metal/slowmetaltrashvideo12.mp4"
output_video_path = "/content/drive/MyDrive/Garbagedata/Trash Videos/outputs_trackclass/slowmetaltrashvideo12_op.mp4"
save_crops_dir = "/content/drive/MyDrive/Garbagedata/Trash Videos/outputs_trackclass/crops_classified/"

# ROI + confidence
roi_top = 0.10
roi_bottom = 0.70
conf = 0.1
classify_conf = 0.1  # Lower threshold for classification to catch more detections

# Run the script
results = track_and_classify_realtime(
    tracking_weights_path=tracking_weights,
    classification_weights_path=classification_weights,
    video_path=video_path,
    output_video_path=output_video_path,
    roi_top=roi_top,
    roi_bottom=roi_bottom,
    conf_thres=conf,
    classify_conf_thres=classify_conf,
    save_crops_dir=save_crops_dir,
    filter_hands=True,
    hand_conf_threshold=0.3,
)

print(f"\n🏆 FINAL RESULT: {results['final_class']}")
print(f"   Metal: {results['metal_count']} | Plastic: {results['plastic_count']}")

Loading tracking model...
Loading classification model...
✓ Classification model loaded (Task: detect)
Loading pose model for hand detection...

🎬 Starting tracking and classification...
✋ Hand filtering enabled - detections overlapping with hands will be excluded
Processed 30 frames...
Processed 60 frames...

📊 CLASSIFICATION RESULTS

🎯 FINAL VERDICT: METAL

Total classifications: 12
  • METAL:     12 (100.0%)
  • PLASTIC:    0 (  0.0%)

📦 Per-Track Classifications:

✅ Saved tracked video to: /content/drive/MyDrive/Garbagedata/Trash Videos/outputs_trackclass/slowmetaltrashvideo12_op.mp4
✅ Saved crops to: /content/drive/MyDrive/Garbagedata/Trash Videos/outputs_trackclass/crops_classified/

🏆 FINAL RESULT: METAL
   Metal: 12 | Plastic: 0


In [ ]:
import os
import torch
from collections import Counter

# Monkey patch torch.load ONCE to force weights_only=False
if not hasattr(torch, '_load_patched'):
    _original_load = torch.load
    def patched_load(*args, **kwargs):
        kwargs['weights_only'] = False
        return _original_load(*args, **kwargs)
    torch.load = patched_load
    torch._load_patched = True

import cv2
from ultralytics import YOLO


def track_and_classify_realtime(
    tracking_weights_path,
    classification_weights_path,
    video_path,
    output_video_path,
    roi_top=0.10,
    roi_bottom=0.70,
    conf_thres=0.4,
    classify_conf_thres=0.1,
    save_crops_dir=None,
    filter_hands=True,
    hand_conf_threshold=0.3,
):
    """
    YOLOv8 tracking + real-time classification with majority voting and hand filtering.

    Args:
        tracking_weights_path: Path to general YOLOv8 model for tracking
        classification_weights_path: Path to your custom trained model (0=metal, 1=plastic)
        video_path: Input video path
        output_video_path: Output video path
        roi_top: Top boundary of ROI (fraction of height)
        roi_bottom: Bottom boundary of ROI (fraction of height)
        conf_thres: Confidence threshold for tracking detections
        classify_conf_thres: Confidence threshold for classification model
        save_crops_dir: Directory to save cropped images
        filter_hands: Whether to filter out hand detections
        hand_conf_threshold: Confidence threshold for hand detection
    """

    # Load models
    print("Loading tracking model...")
    tracking_model = YOLO(tracking_weights_path)

    print("Loading classification model...")
    classification_model = YOLO(classification_weights_path)
    print(f"✓ Classification model loaded (Task: {classification_model.task})")

    # Load pose model for hand detection if filtering is enabled
    hand_model = None
    if filter_hands:
        print("Loading pose model for hand detection...")
        hand_model = YOLO("yolov8n-pose.pt")  # Lightweight pose model

    # Create output dirs
    if output_video_path:
        os.makedirs(os.path.dirname(output_video_path), exist_ok=True)
    if save_crops_dir is not None:
        os.makedirs(save_crops_dir, exist_ok=True)

    # Read video metadata
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    writer = cv2.VideoWriter(
        output_video_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )

    # Precompute ROI vertical boundaries
    y_min = int(roi_top * height)
    y_max = int(roi_bottom * height)

    # Storage for classifications
    all_classifications = []  # List to store all classification results
    track_classifications = {}  # Dict to store classifications per track_id

    # Tracking generator
    results_gen = tracking_model.track(
        source=video_path,
        tracker="bytetrack.yaml",
        conf=conf_thres,
        iou=0.5,
        stream=True,
        verbose=False,
    )

    frame_idx = 0

    print("\n🎬 Starting tracking and classification...")
    if filter_hands:
        print("✋ Hand filtering enabled - detections overlapping with hands will be excluded")

    for result in results_gen:
        frame = result.orig_img.copy()
        h, w = frame.shape[:2]

        # Recalculate ROI if frame shape changes
        y_min = int(roi_top * h)
        y_max = int(roi_bottom * h)

        # Detect hands in the frame if filtering is enabled
        hand_regions = []
        if filter_hands and hand_model is not None:
            hand_results = hand_model(frame, verbose=False)

            if len(hand_results) > 0 and hand_results[0].keypoints is not None:
                keypoints = hand_results[0].keypoints

                # Extract hand keypoints (wrists and hand keypoints)
                # YOLOv8-pose has 17 keypoints: 9=left_wrist, 10=right_wrist
                for person_kpts in keypoints.data:
                    if len(person_kpts) > 0:
                        # Get wrist keypoints (indices 9 and 10)
                        left_wrist = person_kpts[9][:2].cpu().numpy()  # x, y
                        right_wrist = person_kpts[10][:2].cpu().numpy()  # x, y

                        # Create expanded regions around wrists (approximate hand size)
                        hand_size = 100  # pixels radius around wrist

                        for wrist in [left_wrist, right_wrist]:
                            if wrist[0] > 0 and wrist[1] > 0:  # Valid keypoint
                                x1 = int(max(0, wrist[0] - hand_size))
                                y1 = int(max(0, wrist[1] - hand_size))
                                x2 = int(min(w, wrist[0] + hand_size))
                                y2 = int(min(h, wrist[1] + hand_size))
                                hand_regions.append((x1, y1, x2, y2))

                                # Draw hand region for visualization (optional)
                                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 1)
                                cv2.putText(frame, "HAND", (x1, y1-5),
                                          cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)

        # Draw ROI
        cv2.rectangle(frame, (0, y_min), (w, y_max), (0, 255, 255), 2)

        boxes = result.boxes

        if boxes is not None and len(boxes) > 0:
            xyxy = boxes.xyxy.cpu().numpy()
            confs = boxes.conf.cpu().numpy()
            ids = (
                boxes.id.cpu().numpy()
                if boxes.id is not None
                else [-1] * len(xyxy)
            )
            clss = boxes.cls.cpu().numpy()

            for box, score, track_id, cls_id in zip(xyxy, confs, ids, clss):
                if score < conf_thres:
                    continue

                # Skip person class detections (COCO class 0)
                if int(cls_id) == 0:
                    continue

                x1, y1, x2, y2 = box.astype(int)
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2

                # Reject detections outside ROI
                if not (y_min <= cy <= y_max):
                    continue

                # Check if detection overlaps with any hand region
                if filter_hands and hand_regions:
                    overlaps_hand = False
                    for hx1, hy1, hx2, hy2 in hand_regions:
                        # Check if bounding boxes overlap
                        if not (x2 < hx1 or x1 > hx2 or y2 < hy1 or y1 > hy2):
                            # Calculate intersection area
                            ix1 = max(x1, hx1)
                            iy1 = max(y1, hy1)
                            ix2 = min(x2, hx2)
                            iy2 = min(y2, hy2)

                            intersection_area = (ix2 - ix1) * (iy2 - iy1)
                            detection_area = (x2 - x1) * (y2 - y1)

                            # If more than 30% of detection overlaps with hand, skip it
                            if intersection_area / detection_area > 0.3:
                                overlaps_hand = True
                                break

                    if overlaps_hand:
                        # Draw rejected detection in red
                        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 1)
                        cv2.putText(frame, "HAND FILTERED", (x1, y1-5),
                                  cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)
                        continue

                # Crop the detection
                crop = frame[max(0, y1):min(h, y2), max(0, x1):min(w, x2)]

                if crop.size > 0:
                    # Default values
                    class_name = "UNKNOWN"
                    box_color = (0, 255, 0)
                    confidence = 0.0

                    # Classify the crop using custom model
                    classify_results = classification_model(crop, verbose=False, conf=classify_conf_thres)

                    if len(classify_results) > 0:
                        classify_result = classify_results[0]

                        # Check if it's a classification model
                        if hasattr(classify_result, 'probs') and classify_result.probs is not None:
                            probs = classify_result.probs.data.cpu().numpy()
                            predicted_class = int(probs.argmax())
                            confidence = float(probs.max())

                            class_name = "METAL" if predicted_class == 0 else "PLASTIC"
                            box_color = (255, 0, 0) if predicted_class == 0 else (0, 255, 255)

                            # Store classification
                            all_classifications.append(predicted_class)
                            if track_id != -1:
                                if track_id not in track_classifications:
                                    track_classifications[track_id] = []
                                track_classifications[track_id].append(predicted_class)

                        # Check if it's a detection model
                        elif hasattr(classify_result, 'boxes') and classify_result.boxes is not None:
                            classify_boxes = classify_result.boxes

                            if len(classify_boxes) > 0:
                                # Take highest confidence detection
                                best_idx = classify_boxes.conf.argmax()
                                predicted_class = int(classify_boxes.cls[best_idx].cpu().numpy())
                                confidence = float(classify_boxes.conf[best_idx].cpu().numpy())

                                class_name = "METAL" if predicted_class == 0 else "PLASTIC"
                                box_color = (255, 0, 0) if predicted_class == 0 else (0, 255, 255)

                                # Store classification
                                all_classifications.append(predicted_class)
                                if track_id != -1:
                                    if track_id not in track_classifications:
                                        track_classifications[track_id] = []
                                    track_classifications[track_id].append(predicted_class)
                            else:
                                class_name = "NO_DETECTION"

                    # Draw bounding box and label
                    label = f"ID:{int(track_id)} {class_name} {confidence:.2f}"
                    cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)
                    cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)
                    cv2.putText(
                        frame,
                        label,
                        (x1, max(0, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.5,
                        box_color,
                        2,
                        cv2.LINE_AA,
                    )

                    # Save crop with classification
                    if save_crops_dir is not None and track_id != -1 and class_name != "UNKNOWN":
                        crop_path = os.path.join(
                            save_crops_dir,
                            f"frame{frame_idx:05d}_id{int(track_id)}_{class_name}_{confidence:.3f}.jpg",
                        )
                        cv2.imwrite(crop_path, crop)

        # Add overall statistics overlay to video
        if all_classifications:
            class_counts = Counter(all_classifications)
            metal_count = class_counts.get(0, 0)
            plastic_count = class_counts.get(1, 0)
            total = len(all_classifications)

            # Determine current majority
            current_majority = "METAL" if metal_count > plastic_count else "PLASTIC"
            majority_color = (255, 0, 0) if metal_count > plastic_count else (0, 255, 255)

            # Create semi-transparent overlay for statistics
            overlay = frame.copy()
            cv2.rectangle(overlay, (10, 10), (400, 120), (0, 0, 0), -1)
            cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

            # Add text
            cv2.putText(frame, f"CURRENT VERDICT: {current_majority}", (20, 35),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, majority_color, 2, cv2.LINE_AA)
            cv2.putText(frame, f"Metal: {metal_count} ({metal_count/total*100:.1f}%)", (20, 65),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f"Plastic: {plastic_count} ({plastic_count/total*100:.1f}%)", (20, 95),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2, cv2.LINE_AA)

        writer.write(frame)
        frame_idx += 1

        if frame_idx % 30 == 0:
            print(f"Processed {frame_idx} frames...")

    writer.release()

    # Calculate final results
    print("\n" + "="*60)
    print("📊 CLASSIFICATION RESULTS")
    print("="*60)

    final_class_name = None
    metal_count = 0
    plastic_count = 0
    total = 0

    if all_classifications:
        class_counts = Counter(all_classifications)
        total = len(all_classifications)
        metal_count = class_counts.get(0, 0)
        plastic_count = class_counts.get(1, 0)

        metal_percent = (metal_count / total) * 100
        plastic_percent = (plastic_count / total) * 100

        final_class = 0 if metal_count > plastic_count else 1
        final_class_name = "METAL" if final_class == 0 else "PLASTIC"

        print(f"\n🎯 FINAL VERDICT: {final_class_name}")
        print(f"\nTotal classifications: {total}")
        print(f"  • METAL:   {metal_count:4d} ({metal_percent:5.1f}%)")
        print(f"  • PLASTIC: {plastic_count:4d} ({plastic_percent:5.1f}%)")

        # Per-track results
        print(f"\n📦 Per-Track Classifications:")
        for track_id, classifications in sorted(track_classifications.items()):
            track_counts = Counter(classifications)
            track_total = len(classifications)
            track_metal = track_counts.get(0, 0)
            track_plastic = track_counts.get(1, 0)
            track_final = 0 if track_metal > track_plastic else 1
            track_final_name = "METAL" if track_final == 0 else "PLASTIC"

            print(f"  Track ID {int(track_id):3d}: {track_final_name:7s} "
                  f"(M:{track_metal:3d}, P:{track_plastic:3d}, Total:{track_total:3d})")
    else:
        print("\n⚠️  No classifications made!")

    print("\n" + "="*60)
    print(f"✅ Saved tracked video to: {output_video_path}")
    if save_crops_dir:
        print(f"✅ Saved crops to: {save_crops_dir}")
    print("="*60)

    return {
        'final_class': final_class_name,
        'metal_count': metal_count,
        'plastic_count': plastic_count,
        'total_count': total,
        'track_results': track_classifications
    }


# --------------------------------------------------------------------
# ---------------------- CONFIGURATION -------------------------------
# --------------------------------------------------------------------

# Path to ORIGINAL YOLOv8 model for tracking (general object detection)
tracking_weights = "/content/yolov8m.pt"

# Path to YOUR CUSTOM trained model for classification (metal=0, plastic=1)
classification_weights = "/content/drive/MyDrive/Garbagedata/Garbagedata_Unzipped_Files/Ishita_Model/Garbagedata/Models_V3m_Runs/Model_V3m_Run1/weights/best.pt"

# Video and output paths
video_path = "/content/drive/MyDrive/Garbagedata/Plainbackground_videos/metal/slowmetalglassvideo2.mp4"
output_video_path = "/content/drive/MyDrive/Garbagedata/Trash Videos/outputs_trackclass/slowmetalglassvideo2_op.mp4"
save_crops_dir = "/content/drive/MyDrive/Garbagedata/Trash Videos/outputs_trackclass/crops_classified/"

# ROI + confidence
roi_top = 0.10
roi_bottom = 0.70
conf = 0.1
classify_conf = 0.1  # Lower threshold for classification to catch more detections

# Run the script
results = track_and_classify_realtime(
    tracking_weights_path=tracking_weights,
    classification_weights_path=classification_weights,
    video_path=video_path,
    output_video_path=output_video_path,
    roi_top=roi_top,
    roi_bottom=roi_bottom,
    conf_thres=conf,
    classify_conf_thres=classify_conf,
    save_crops_dir=save_crops_dir,
    filter_hands=True,
    hand_conf_threshold=0.3,
)

print(f"\n🏆 FINAL RESULT: {results['final_class']}")
print(f"   Metal: {results['metal_count']} | Plastic: {results['plastic_count']}")

Loading tracking model...
Loading classification model...
✓ Classification model loaded (Task: detect)
Loading pose model for hand detection...

🎬 Starting tracking and classification...
✋ Hand filtering enabled - detections overlapping with hands will be excluded
Processed 30 frames...
Processed 60 frames...
Processed 90 frames...

📊 CLASSIFICATION RESULTS

⚠️  No classifications made!

✅ Saved tracked video to: /content/drive/MyDrive/Garbagedata/Trash Videos/outputs_trackclass/slowmetalglassvideo2_op.mp4
✅ Saved crops to: /content/drive/MyDrive/Garbagedata/Trash Videos/outputs_trackclass/crops_classified/

🏆 FINAL RESULT: None
   Metal: 0 | Plastic: 0


PLASTIC

In [ ]:
import os
import torch
from collections import Counter

# Monkey patch torch.load ONCE to force weights_only=False
if not hasattr(torch, '_load_patched'):
    _original_load = torch.load
    def patched_load(*args, **kwargs):
        kwargs['weights_only'] = False
        return _original_load(*args, **kwargs)
    torch.load = patched_load
    torch._load_patched = True

import cv2
from ultralytics import YOLO


def track_and_classify_realtime(
    tracking_weights_path,
    classification_weights_path,
    video_path,
    output_video_path,
    roi_top=0.10,
    roi_bottom=0.70,
    conf_thres=0.4,
    classify_conf_thres=0.1,
    save_crops_dir=None,
    filter_hands=True,
    hand_conf_threshold=0.3,
):
    """
    YOLOv8 tracking + real-time classification with majority voting and hand filtering.

    Args:
        tracking_weights_path: Path to general YOLOv8 model for tracking
        classification_weights_path: Path to your custom trained model (0=metal, 1=plastic)
        video_path: Input video path
        output_video_path: Output video path
        roi_top: Top boundary of ROI (fraction of height)
        roi_bottom: Bottom boundary of ROI (fraction of height)
        conf_thres: Confidence threshold for tracking detections
        classify_conf_thres: Confidence threshold for classification model
        save_crops_dir: Directory to save cropped images
        filter_hands: Whether to filter out hand detections
        hand_conf_threshold: Confidence threshold for hand detection
    """

    # Load models
    print("Loading tracking model...")
    tracking_model = YOLO(tracking_weights_path)

    print("Loading classification model...")
    classification_model = YOLO(classification_weights_path)
    print(f"✓ Classification model loaded (Task: {classification_model.task})")

    # Load pose model for hand detection if filtering is enabled
    hand_model = None
    if filter_hands:
        print("Loading pose model for hand detection...")
        hand_model = YOLO("yolov8n-pose.pt")  # Lightweight pose model

    # Create output dirs
    if output_video_path:
        os.makedirs(os.path.dirname(output_video_path), exist_ok=True)
    if save_crops_dir is not None:
        os.makedirs(save_crops_dir, exist_ok=True)

    # Read video metadata
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    writer = cv2.VideoWriter(
        output_video_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )

    # Precompute ROI vertical boundaries
    y_min = int(roi_top * height)
    y_max = int(roi_bottom * height)

    # Storage for classifications
    all_classifications = []  # List to store all classification results
    track_classifications = {}  # Dict to store classifications per track_id

    # Tracking generator
    results_gen = tracking_model.track(
        source=video_path,
        tracker="bytetrack.yaml",
        conf=conf_thres,
        iou=0.5,
        stream=True,
        verbose=False,
    )

    frame_idx = 0

    print("\n🎬 Starting tracking and classification...")
    if filter_hands:
        print("✋ Hand filtering enabled - detections overlapping with hands will be excluded")

    for result in results_gen:
        frame = result.orig_img.copy()
        h, w = frame.shape[:2]

        # Recalculate ROI if frame shape changes
        y_min = int(roi_top * h)
        y_max = int(roi_bottom * h)

        # Detect hands in the frame if filtering is enabled
        hand_regions = []
        if filter_hands and hand_model is not None:
            hand_results = hand_model(frame, verbose=False)

            if len(hand_results) > 0 and hand_results[0].keypoints is not None:
                keypoints = hand_results[0].keypoints

                # Extract hand keypoints (wrists and hand keypoints)
                # YOLOv8-pose has 17 keypoints: 9=left_wrist, 10=right_wrist
                for person_kpts in keypoints.data:
                    if len(person_kpts) > 0:
                        # Get wrist keypoints (indices 9 and 10)
                        left_wrist = person_kpts[9][:2].cpu().numpy()  # x, y
                        right_wrist = person_kpts[10][:2].cpu().numpy()  # x, y

                        # Create expanded regions around wrists (approximate hand size)
                        hand_size = 100  # pixels radius around wrist

                        for wrist in [left_wrist, right_wrist]:
                            if wrist[0] > 0 and wrist[1] > 0:  # Valid keypoint
                                x1 = int(max(0, wrist[0] - hand_size))
                                y1 = int(max(0, wrist[1] - hand_size))
                                x2 = int(min(w, wrist[0] + hand_size))
                                y2 = int(min(h, wrist[1] + hand_size))
                                hand_regions.append((x1, y1, x2, y2))

                                # Draw hand region for visualization (optional)
                                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 1)
                                cv2.putText(frame, "HAND", (x1, y1-5),
                                          cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)

        # Draw ROI
        cv2.rectangle(frame, (0, y_min), (w, y_max), (0, 255, 255), 2)

        boxes = result.boxes

        if boxes is not None and len(boxes) > 0:
            xyxy = boxes.xyxy.cpu().numpy()
            confs = boxes.conf.cpu().numpy()
            ids = (
                boxes.id.cpu().numpy()
                if boxes.id is not None
                else [-1] * len(xyxy)
            )
            clss = boxes.cls.cpu().numpy()

            for box, score, track_id, cls_id in zip(xyxy, confs, ids, clss):
                if score < conf_thres:
                    continue

                # Skip person class detections (COCO class 0)
                if int(cls_id) == 0:
                    continue

                x1, y1, x2, y2 = box.astype(int)
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2

                # Reject detections outside ROI
                if not (y_min <= cy <= y_max):
                    continue

                # Check if detection overlaps with any hand region
                if filter_hands and hand_regions:
                    overlaps_hand = False
                    for hx1, hy1, hx2, hy2 in hand_regions:
                        # Check if bounding boxes overlap
                        if not (x2 < hx1 or x1 > hx2 or y2 < hy1 or y1 > hy2):
                            # Calculate intersection area
                            ix1 = max(x1, hx1)
                            iy1 = max(y1, hy1)
                            ix2 = min(x2, hx2)
                            iy2 = min(y2, hy2)

                            intersection_area = (ix2 - ix1) * (iy2 - iy1)
                            detection_area = (x2 - x1) * (y2 - y1)

                            # If more than 30% of detection overlaps with hand, skip it
                            if intersection_area / detection_area > 0.3:
                                overlaps_hand = True
                                break

                    if overlaps_hand:
                        # Draw rejected detection in red
                        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 1)
                        cv2.putText(frame, "HAND FILTERED", (x1, y1-5),
                                  cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)
                        continue

                # Crop the detection
                crop = frame[max(0, y1):min(h, y2), max(0, x1):min(w, x2)]

                if crop.size > 0:
                    # Default values
                    class_name = "UNKNOWN"
                    box_color = (0, 255, 0)
                    confidence = 0.0

                    # Classify the crop using custom model
                    classify_results = classification_model(crop, verbose=False, conf=classify_conf_thres)

                    if len(classify_results) > 0:
                        classify_result = classify_results[0]

                        # Check if it's a classification model
                        if hasattr(classify_result, 'probs') and classify_result.probs is not None:
                            probs = classify_result.probs.data.cpu().numpy()
                            predicted_class = int(probs.argmax())
                            confidence = float(probs.max())

                            class_name = "METAL" if predicted_class == 0 else "PLASTIC"
                            box_color = (255, 0, 0) if predicted_class == 0 else (0, 255, 255)

                            # Store classification
                            all_classifications.append(predicted_class)
                            if track_id != -1:
                                if track_id not in track_classifications:
                                    track_classifications[track_id] = []
                                track_classifications[track_id].append(predicted_class)

                        # Check if it's a detection model
                        elif hasattr(classify_result, 'boxes') and classify_result.boxes is not None:
                            classify_boxes = classify_result.boxes

                            if len(classify_boxes) > 0:
                                # Take highest confidence detection
                                best_idx = classify_boxes.conf.argmax()
                                predicted_class = int(classify_boxes.cls[best_idx].cpu().numpy())
                                confidence = float(classify_boxes.conf[best_idx].cpu().numpy())

                                class_name = "METAL" if predicted_class == 0 else "PLASTIC"
                                box_color = (255, 0, 0) if predicted_class == 0 else (0, 255, 255)

                                # Store classification
                                all_classifications.append(predicted_class)
                                if track_id != -1:
                                    if track_id not in track_classifications:
                                        track_classifications[track_id] = []
                                    track_classifications[track_id].append(predicted_class)
                            else:
                                class_name = "NO_DETECTION"

                    # Draw bounding box and label
                    label = f"ID:{int(track_id)} {class_name} {confidence:.2f}"
                    cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)
                    cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)
                    cv2.putText(
                        frame,
                        label,
                        (x1, max(0, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.5,
                        box_color,
                        2,
                        cv2.LINE_AA,
                    )

                    # Save crop with classification
                    if save_crops_dir is not None and track_id != -1 and class_name != "UNKNOWN":
                        crop_path = os.path.join(
                            save_crops_dir,
                            f"frame{frame_idx:05d}_id{int(track_id)}_{class_name}_{confidence:.3f}.jpg",
                        )
                        cv2.imwrite(crop_path, crop)

        # Add overall statistics overlay to video
        if all_classifications:
            class_counts = Counter(all_classifications)
            metal_count = class_counts.get(0, 0)
            plastic_count = class_counts.get(1, 0)
            total = len(all_classifications)

            # Determine current majority
            current_majority = "METAL" if metal_count > plastic_count else "PLASTIC"
            majority_color = (255, 0, 0) if metal_count > plastic_count else (0, 255, 255)

            # Create semi-transparent overlay for statistics
            overlay = frame.copy()
            cv2.rectangle(overlay, (10, 10), (400, 120), (0, 0, 0), -1)
            cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

            # Add text
            cv2.putText(frame, f"CURRENT VERDICT: {current_majority}", (20, 35),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, majority_color, 2, cv2.LINE_AA)
            cv2.putText(frame, f"Metal: {metal_count} ({metal_count/total*100:.1f}%)", (20, 65),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f"Plastic: {plastic_count} ({plastic_count/total*100:.1f}%)", (20, 95),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2, cv2.LINE_AA)

        writer.write(frame)
        frame_idx += 1

        if frame_idx % 30 == 0:
            print(f"Processed {frame_idx} frames...")

    writer.release()

    # Calculate final results
    print("\n" + "="*60)
    print("📊 CLASSIFICATION RESULTS")
    print("="*60)

    final_class_name = None
    metal_count = 0
    plastic_count = 0
    total = 0

    if all_classifications:
        class_counts = Counter(all_classifications)
        total = len(all_classifications)
        metal_count = class_counts.get(0, 0)
        plastic_count = class_counts.get(1, 0)

        metal_percent = (metal_count / total) * 100
        plastic_percent = (plastic_count / total) * 100

        final_class = 0 if metal_count > plastic_count else 1
        final_class_name = "METAL" if final_class == 0 else "PLASTIC"

        print(f"\n🎯 FINAL VERDICT: {final_class_name}")
        print(f"\nTotal classifications: {total}")
        print(f"  • METAL:   {metal_count:4d} ({metal_percent:5.1f}%)")
        print(f"  • PLASTIC: {plastic_count:4d} ({plastic_percent:5.1f}%)")

        # Per-track results
        print(f"\n📦 Per-Track Classifications:")
        for track_id, classifications in sorted(track_classifications.items()):
            track_counts = Counter(classifications)
            track_total = len(classifications)
            track_metal = track_counts.get(0, 0)
            track_plastic = track_counts.get(1, 0)
            track_final = 0 if track_metal > track_plastic else 1
            track_final_name = "METAL" if track_final == 0 else "PLASTIC"

            print(f"  Track ID {int(track_id):3d}: {track_final_name:7s} "
                  f"(M:{track_metal:3d}, P:{track_plastic:3d}, Total:{track_total:3d})")
    else:
        print("\n⚠️  No classifications made!")

    print("\n" + "="*60)
    print(f"✅ Saved tracked video to: {output_video_path}")
    if save_crops_dir:
        print(f"✅ Saved crops to: {save_crops_dir}")
    print("="*60)

    return {
        'final_class': final_class_name,
        'metal_count': metal_count,
        'plastic_count': plastic_count,
        'total_count': total,
        'track_results': track_classifications
    }


# --------------------------------------------------------------------
# ---------------------- CONFIGURATION -------------------------------
# --------------------------------------------------------------------

# Path to ORIGINAL YOLOv8 model for tracking (general object detection)
tracking_weights = "/content/yolov8m.pt"

# Path to YOUR CUSTOM trained model for classification (metal=0, plastic=1)
classification_weights = "/content/drive/MyDrive/Garbagedata/Garbagedata_Unzipped_Files/Ishita_Model/Garbagedata/Models_V3m_Runs/Model_V3m_Run1/weights/best.pt"

# Video and output paths
video_path = "/content/drive/MyDrive/Garbagedata/Plainbackground_videos/metal/trashvideo_trimeslowed.mp4"
output_video_path = "/content/drive/MyDrive/Garbagedata/Plainbackground_videos/metal/metal_outputs_trackclass/trashvideo_trimeslowed_op512_c1.mp4"
save_crops_dir = "/content/drive/MyDrive/Garbagedata/Plainbackground_videos/metal/metal_outputs_trackclass/crops_classified/"

# ROI + confidence
roi_top = 0.10
roi_bottom = 0.70
conf = 0.4
classify_conf = 0.1  # Lower threshold for classification to catch more detections

# Run the script
results = track_and_classify_realtime(
    tracking_weights_path=tracking_weights,
    classification_weights_path=classification_weights,
    video_path=video_path,
    output_video_path=output_video_path,
    roi_top=roi_top,
    roi_bottom=roi_bottom,
    conf_thres=conf,
    classify_conf_thres=classify_conf,
    save_crops_dir=save_crops_dir,
    filter_hands=True,
    hand_conf_threshold=0.3,
)

print(f"\n🏆 FINAL RESULT: {results['final_class']}")
print(f"   Metal: {results['metal_count']} | Plastic: {results['plastic_count']}")

Loading tracking model...
Loading classification model...
✓ Classification model loaded (Task: detect)
Loading pose model for hand detection...

🎬 Starting tracking and classification...
✋ Hand filtering enabled - detections overlapping with hands will be excluded
Processed 30 frames...
Processed 60 frames...
Processed 90 frames...
Processed 120 frames...
Processed 150 frames...
Processed 180 frames...

📊 CLASSIFICATION RESULTS

🎯 FINAL VERDICT: PLASTIC

Total classifications: 98
  • METAL:      0 (  0.0%)
  • PLASTIC:   98 (100.0%)

📦 Per-Track Classifications:
  Track ID   1: PLASTIC (M:  0, P: 98, Total: 98)

✅ Saved tracked video to: /content/drive/MyDrive/Garbagedata/Plainbackground_videos/metal/metal_outputs_trackclass/trashvideo_trimeslowed_op512_c1.mp4
✅ Saved crops to: /content/drive/MyDrive/Garbagedata/Plainbackground_videos/metal/metal_outputs_trackclass/crops_classified/

🏆 FINAL RESULT: PLASTIC
   Metal: 0 | Plastic: 98


In [ ]:
import os
import torch
from collections import Counter

# Monkey patch torch.load ONCE to force weights_only=False
if not hasattr(torch, '_load_patched'):
    _original_load = torch.load
    def patched_load(*args, **kwargs):
        kwargs['weights_only'] = False
        return _original_load(*args, **kwargs)
    torch.load = patched_load
    torch._load_patched = True

import cv2
from ultralytics import YOLO


def track_and_classify_realtime(
    tracking_weights_path,
    classification_weights_path,
    video_path,
    output_video_path,
    roi_top=0.10,
    roi_bottom=0.70,
    conf_thres=0.4,
    classify_conf_thres=0.1,
    save_crops_dir=None,
    filter_hands=True,
    hand_conf_threshold=0.3,
):
    """
    YOLOv8 tracking + real-time classification with majority voting and hand filtering.

    Args:
        tracking_weights_path: Path to general YOLOv8 model for tracking
        classification_weights_path: Path to your custom trained model (0=metal, 1=plastic)
        video_path: Input video path
        output_video_path: Output video path
        roi_top: Top boundary of ROI (fraction of height)
        roi_bottom: Bottom boundary of ROI (fraction of height)
        conf_thres: Confidence threshold for tracking detections
        classify_conf_thres: Confidence threshold for classification model
        save_crops_dir: Directory to save cropped images
        filter_hands: Whether to filter out hand detections
        hand_conf_threshold: Confidence threshold for hand detection
    """

    # Load models
    print("Loading tracking model...")
    tracking_model = YOLO(tracking_weights_path)

    print("Loading classification model...")
    classification_model = YOLO(classification_weights_path)
    print(f"✓ Classification model loaded (Task: {classification_model.task})")

    # Load pose model for hand detection if filtering is enabled
    hand_model = None
    if filter_hands:
        print("Loading pose model for hand detection...")
        hand_model = YOLO("yolov8n-pose.pt")  # Lightweight pose model

    # Create output dirs
    if output_video_path:
        os.makedirs(os.path.dirname(output_video_path), exist_ok=True)
    if save_crops_dir is not None:
        os.makedirs(save_crops_dir, exist_ok=True)

    # Read video metadata
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    writer = cv2.VideoWriter(
        output_video_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )

    # Precompute ROI vertical boundaries
    y_min = int(roi_top * height)
    y_max = int(roi_bottom * height)

    # Storage for classifications
    all_classifications = []  # List to store all classification results
    track_classifications = {}  # Dict to store classifications per track_id

    # Tracking generator
    results_gen = tracking_model.track(
        source=video_path,
        tracker="bytetrack.yaml",
        conf=conf_thres,
        iou=0.5,
        stream=True,
        verbose=False,
    )

    frame_idx = 0

    print("\n🎬 Starting tracking and classification...")
    if filter_hands:
        print("✋ Hand filtering enabled - detections overlapping with hands will be excluded")

    for result in results_gen:
        frame = result.orig_img.copy()
        h, w = frame.shape[:2]

        # Recalculate ROI if frame shape changes
        y_min = int(roi_top * h)
        y_max = int(roi_bottom * h)

        # Detect hands in the frame if filtering is enabled
        hand_regions = []
        if filter_hands and hand_model is not None:
            hand_results = hand_model(frame, verbose=False)

            if len(hand_results) > 0 and hand_results[0].keypoints is not None:
                keypoints = hand_results[0].keypoints

                # Extract hand keypoints (wrists and hand keypoints)
                # YOLOv8-pose has 17 keypoints: 9=left_wrist, 10=right_wrist
                for person_kpts in keypoints.data:
                    if len(person_kpts) > 0:
                        # Get wrist keypoints (indices 9 and 10)
                        left_wrist = person_kpts[9][:2].cpu().numpy()  # x, y
                        right_wrist = person_kpts[10][:2].cpu().numpy()  # x, y

                        # Create expanded regions around wrists (approximate hand size)
                        hand_size = 100  # pixels radius around wrist

                        for wrist in [left_wrist, right_wrist]:
                            if wrist[0] > 0 and wrist[1] > 0:  # Valid keypoint
                                x1 = int(max(0, wrist[0] - hand_size))
                                y1 = int(max(0, wrist[1] - hand_size))
                                x2 = int(min(w, wrist[0] + hand_size))
                                y2 = int(min(h, wrist[1] + hand_size))
                                hand_regions.append((x1, y1, x2, y2))

                                # Draw hand region for visualization (optional)
                                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 1)
                                cv2.putText(frame, "HAND", (x1, y1-5),
                                          cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)

        # Draw ROI
        cv2.rectangle(frame, (0, y_min), (w, y_max), (0, 255, 255), 2)

        boxes = result.boxes

        if boxes is not None and len(boxes) > 0:
            xyxy = boxes.xyxy.cpu().numpy()
            confs = boxes.conf.cpu().numpy()
            ids = (
                boxes.id.cpu().numpy()
                if boxes.id is not None
                else [-1] * len(xyxy)
            )
            clss = boxes.cls.cpu().numpy()

            for box, score, track_id, cls_id in zip(xyxy, confs, ids, clss):
                if score < conf_thres:
                    continue

                # Skip person class detections (COCO class 0)
                if int(cls_id) == 0:
                    continue

                x1, y1, x2, y2 = box.astype(int)
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2

                # Reject detections outside ROI
                if not (y_min <= cy <= y_max):
                    continue

                # Check if detection overlaps with any hand region
                if filter_hands and hand_regions:
                    overlaps_hand = False
                    for hx1, hy1, hx2, hy2 in hand_regions:
                        # Check if bounding boxes overlap
                        if not (x2 < hx1 or x1 > hx2 or y2 < hy1 or y1 > hy2):
                            # Calculate intersection area
                            ix1 = max(x1, hx1)
                            iy1 = max(y1, hy1)
                            ix2 = min(x2, hx2)
                            iy2 = min(y2, hy2)

                            intersection_area = (ix2 - ix1) * (iy2 - iy1)
                            detection_area = (x2 - x1) * (y2 - y1)

                            # If more than 30% of detection overlaps with hand, skip it
                            if intersection_area / detection_area > 0.3:
                                overlaps_hand = True
                                break

                    if overlaps_hand:
                        # Draw rejected detection in red
                        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 1)
                        cv2.putText(frame, "HAND FILTERED", (x1, y1-5),
                                  cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)
                        continue

                # Crop the detection
                crop = frame[max(0, y1):min(h, y2), max(0, x1):min(w, x2)]

                if crop.size > 0:
                    # Default values
                    class_name = "UNKNOWN"
                    box_color = (0, 255, 0)
                    confidence = 0.0

                    # Classify the crop using custom model
                    classify_results = classification_model(crop, verbose=False, conf=classify_conf_thres)

                    if len(classify_results) > 0:
                        classify_result = classify_results[0]

                        # Check if it's a classification model
                        if hasattr(classify_result, 'probs') and classify_result.probs is not None:
                            probs = classify_result.probs.data.cpu().numpy()
                            predicted_class = int(probs.argmax())
                            confidence = float(probs.max())

                            class_name = "METAL" if predicted_class == 0 else "PLASTIC"
                            box_color = (255, 0, 0) if predicted_class == 0 else (0, 255, 255)

                            # Store classification
                            all_classifications.append(predicted_class)
                            if track_id != -1:
                                if track_id not in track_classifications:
                                    track_classifications[track_id] = []
                                track_classifications[track_id].append(predicted_class)

                        # Check if it's a detection model
                        elif hasattr(classify_result, 'boxes') and classify_result.boxes is not None:
                            classify_boxes = classify_result.boxes

                            if len(classify_boxes) > 0:
                                # Take highest confidence detection
                                best_idx = classify_boxes.conf.argmax()
                                predicted_class = int(classify_boxes.cls[best_idx].cpu().numpy())
                                confidence = float(classify_boxes.conf[best_idx].cpu().numpy())

                                class_name = "METAL" if predicted_class == 0 else "PLASTIC"
                                box_color = (255, 0, 0) if predicted_class == 0 else (0, 255, 255)

                                # Store classification
                                all_classifications.append(predicted_class)
                                if track_id != -1:
                                    if track_id not in track_classifications:
                                        track_classifications[track_id] = []
                                    track_classifications[track_id].append(predicted_class)
                            else:
                                class_name = "NO_DETECTION"

                    # Draw bounding box and label
                    label = f"ID:{int(track_id)} {class_name} {confidence:.2f}"
                    cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)
                    cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)
                    cv2.putText(
                        frame,
                        label,
                        (x1, max(0, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.5,
                        box_color,
                        2,
                        cv2.LINE_AA,
                    )

                    # Save crop with classification
                    if save_crops_dir is not None and track_id != -1 and class_name != "UNKNOWN":
                        crop_path = os.path.join(
                            save_crops_dir,
                            f"frame{frame_idx:05d}_id{int(track_id)}_{class_name}_{confidence:.3f}.jpg",
                        )
                        cv2.imwrite(crop_path, crop)

        # Add overall statistics overlay to video
        if all_classifications:
            class_counts = Counter(all_classifications)
            metal_count = class_counts.get(0, 0)
            plastic_count = class_counts.get(1, 0)
            total = len(all_classifications)

            # Determine current majority
            current_majority = "METAL" if metal_count > plastic_count else "PLASTIC"
            majority_color = (255, 0, 0) if metal_count > plastic_count else (0, 255, 255)

            # Create semi-transparent overlay for statistics
            overlay = frame.copy()
            cv2.rectangle(overlay, (10, 10), (400, 120), (0, 0, 0), -1)
            cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

            # Add text
            cv2.putText(frame, f"CURRENT VERDICT: {current_majority}", (20, 35),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, majority_color, 2, cv2.LINE_AA)
            cv2.putText(frame, f"Metal: {metal_count} ({metal_count/total*100:.1f}%)", (20, 65),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f"Plastic: {plastic_count} ({plastic_count/total*100:.1f}%)", (20, 95),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2, cv2.LINE_AA)

        writer.write(frame)
        frame_idx += 1

        if frame_idx % 30 == 0:
            print(f"Processed {frame_idx} frames...")

    writer.release()

    # Calculate final results
    print("\n" + "="*60)
    print("📊 CLASSIFICATION RESULTS")
    print("="*60)

    final_class_name = None
    metal_count = 0
    plastic_count = 0
    total = 0

    if all_classifications:
        class_counts = Counter(all_classifications)
        total = len(all_classifications)
        metal_count = class_counts.get(0, 0)
        plastic_count = class_counts.get(1, 0)

        metal_percent = (metal_count / total) * 100
        plastic_percent = (plastic_count / total) * 100

        final_class = 0 if metal_count > plastic_count else 1
        final_class_name = "METAL" if final_class == 0 else "PLASTIC"

        print(f"\n🎯 FINAL VERDICT: {final_class_name}")
        print(f"\nTotal classifications: {total}")
        print(f"  • METAL:   {metal_count:4d} ({metal_percent:5.1f}%)")
        print(f"  • PLASTIC: {plastic_count:4d} ({plastic_percent:5.1f}%)")

        # Per-track results
        print(f"\n📦 Per-Track Classifications:")
        for track_id, classifications in sorted(track_classifications.items()):
            track_counts = Counter(classifications)
            track_total = len(classifications)
            track_metal = track_counts.get(0, 0)
            track_plastic = track_counts.get(1, 0)
            track_final = 0 if track_metal > track_plastic else 1
            track_final_name = "METAL" if track_final == 0 else "PLASTIC"

            print(f"  Track ID {int(track_id):3d}: {track_final_name:7s} "
                  f"(M:{track_metal:3d}, P:{track_plastic:3d}, Total:{track_total:3d})")
    else:
        print("\n⚠️  No classifications made!")

    print("\n" + "="*60)
    print(f"✅ Saved tracked video to: {output_video_path}")
    if save_crops_dir:
        print(f"✅ Saved crops to: {save_crops_dir}")
    print("="*60)

    return {
        'final_class': final_class_name,
        'metal_count': metal_count,
        'plastic_count': plastic_count,
        'total_count': total,
        'track_results': track_classifications
    }


# --------------------------------------------------------------------
# ---------------------- CONFIGURATION -------------------------------
# --------------------------------------------------------------------

# Path to ORIGINAL YOLOv8 model for tracking (general object detection)
tracking_weights = "/content/yolov8m.pt"

# Path to YOUR CUSTOM trained model for classification (metal=0, plastic=1)
classification_weights = "/content/drive/MyDrive/Garbagedata/Garbagedata_Unzipped_Files/Ishita_Model/Garbagedata/Models_V3m_Runs/Model_V3m_Run1/weights/best.pt"

# Video and output paths
video_path = "/content/drive/MyDrive/Garbagedata/Plainbackground_videos/plastic/plastictrashvideo1.mp4"
output_video_path = "/content/drive/MyDrive/Garbagedata/Plainbackground_videos/plastic/plastic_outputs_trackclass/plastictrashvideo1.mp4"
save_crops_dir = "/content/drive/MyDrive/Garbagedata/Plainbackground_videos/plastic/plastic_outputs/crops_efficientnet/"

# ROI + confidence
roi_top = 0.10
roi_bottom = 0.70
conf = 0.4
classify_conf = 0.1  # Lower threshold for classification to catch more detections

# Run the script
results = track_and_classify_realtime(
    tracking_weights_path=tracking_weights,
    classification_weights_path=classification_weights,
    video_path=video_path,
    output_video_path=output_video_path,
    roi_top=roi_top,
    roi_bottom=roi_bottom,
    conf_thres=conf,
    classify_conf_thres=classify_conf,
    save_crops_dir=save_crops_dir,
    filter_hands=True,
    hand_conf_threshold=0.3,
)

print(f"\n🏆 FINAL RESULT: {results['final_class']}")
print(f"   Metal: {results['metal_count']} | Plastic: {results['plastic_count']}")

Loading tracking model...
Loading classification model...
✓ Classification model loaded (Task: detect)
Loading pose model for hand detection...

🎬 Starting tracking and classification...
✋ Hand filtering enabled - detections overlapping with hands will be excluded
Processed 30 frames...
Processed 60 frames...
Processed 90 frames...
Processed 120 frames...

📊 CLASSIFICATION RESULTS

🎯 FINAL VERDICT: PLASTIC

Total classifications: 6
  • METAL:      0 (  0.0%)
  • PLASTIC:    6 (100.0%)

📦 Per-Track Classifications:
  Track ID   1: PLASTIC (M:  0, P:  3, Total:  3)

✅ Saved tracked video to: /content/drive/MyDrive/Garbagedata/Plainbackground_videos/plastic/plastic_outputs_trackclass/plastictrashvideo1.mp4
✅ Saved crops to: /content/drive/MyDrive/Garbagedata/Plainbackground_videos/plastic/plastic_outputs/crops_efficientnet/

🏆 FINAL RESULT: PLASTIC
   Metal: 0 | Plastic: 6


In [ ]:
import os
import torch
from collections import Counter

# Monkey patch torch.load ONCE to force weights_only=False
if not hasattr(torch, '_load_patched'):
    _original_load = torch.load
    def patched_load(*args, **kwargs):
        kwargs['weights_only'] = False
        return _original_load(*args, **kwargs)
    torch.load = patched_load
    torch._load_patched = True

import cv2
from ultralytics import YOLO


def track_and_classify_realtime(
    tracking_weights_path,
    classification_weights_path,
    video_path,
    output_video_path,
    roi_top=0.10,
    roi_bottom=0.70,
    conf_thres=0.4,
    classify_conf_thres=0.1,
    save_crops_dir=None,
    filter_hands=True,
    hand_conf_threshold=0.3,
):
    """
    YOLOv8 tracking + real-time classification with majority voting and hand filtering.

    Args:
        tracking_weights_path: Path to general YOLOv8 model for tracking
        classification_weights_path: Path to your custom trained model (0=metal, 1=plastic)
        video_path: Input video path
        output_video_path: Output video path
        roi_top: Top boundary of ROI (fraction of height)
        roi_bottom: Bottom boundary of ROI (fraction of height)
        conf_thres: Confidence threshold for tracking detections
        classify_conf_thres: Confidence threshold for classification model
        save_crops_dir: Directory to save cropped images
        filter_hands: Whether to filter out hand detections
        hand_conf_threshold: Confidence threshold for hand detection
    """

    # Load models
    print("Loading tracking model...")
    tracking_model = YOLO(tracking_weights_path)

    print("Loading classification model...")
    classification_model = YOLO(classification_weights_path)
    print(f"✓ Classification model loaded (Task: {classification_model.task})")

    # Load pose model for hand detection if filtering is enabled
    hand_model = None
    if filter_hands:
        print("Loading pose model for hand detection...")
        hand_model = YOLO("yolov8n-pose.pt")  # Lightweight pose model

    # Create output dirs
    if output_video_path:
        os.makedirs(os.path.dirname(output_video_path), exist_ok=True)
    if save_crops_dir is not None:
        os.makedirs(save_crops_dir, exist_ok=True)

    # Read video metadata
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    writer = cv2.VideoWriter(
        output_video_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )

    # Precompute ROI vertical boundaries
    y_min = int(roi_top * height)
    y_max = int(roi_bottom * height)

    # Storage for classifications
    all_classifications = []  # List to store all classification results
    track_classifications = {}  # Dict to store classifications per track_id

    # Tracking generator
    results_gen = tracking_model.track(
        source=video_path,
        tracker="bytetrack.yaml",
        conf=conf_thres,
        iou=0.5,
        stream=True,
        verbose=False,
    )

    frame_idx = 0

    print("\n🎬 Starting tracking and classification...")
    if filter_hands:
        print("✋ Hand filtering enabled - detections overlapping with hands will be excluded")

    for result in results_gen:
        frame = result.orig_img.copy()
        h, w = frame.shape[:2]

        # Recalculate ROI if frame shape changes
        y_min = int(roi_top * h)
        y_max = int(roi_bottom * h)

        # Detect hands in the frame if filtering is enabled
        hand_regions = []
        if filter_hands and hand_model is not None:
            hand_results = hand_model(frame, verbose=False)

            if len(hand_results) > 0 and hand_results[0].keypoints is not None:
                keypoints = hand_results[0].keypoints

                # Extract hand keypoints (wrists and hand keypoints)
                # YOLOv8-pose has 17 keypoints: 9=left_wrist, 10=right_wrist
                for person_kpts in keypoints.data:
                    if len(person_kpts) > 0:
                        # Get wrist keypoints (indices 9 and 10)
                        left_wrist = person_kpts[9][:2].cpu().numpy()  # x, y
                        right_wrist = person_kpts[10][:2].cpu().numpy()  # x, y

                        # Create expanded regions around wrists (approximate hand size)
                        hand_size = 100  # pixels radius around wrist

                        for wrist in [left_wrist, right_wrist]:
                            if wrist[0] > 0 and wrist[1] > 0:  # Valid keypoint
                                x1 = int(max(0, wrist[0] - hand_size))
                                y1 = int(max(0, wrist[1] - hand_size))
                                x2 = int(min(w, wrist[0] + hand_size))
                                y2 = int(min(h, wrist[1] + hand_size))
                                hand_regions.append((x1, y1, x2, y2))

                                # Draw hand region for visualization (optional)
                                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 1)
                                cv2.putText(frame, "HAND", (x1, y1-5),
                                          cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)

        # Draw ROI
        cv2.rectangle(frame, (0, y_min), (w, y_max), (0, 255, 255), 2)

        boxes = result.boxes

        if boxes is not None and len(boxes) > 0:
            xyxy = boxes.xyxy.cpu().numpy()
            confs = boxes.conf.cpu().numpy()
            ids = (
                boxes.id.cpu().numpy()
                if boxes.id is not None
                else [-1] * len(xyxy)
            )
            clss = boxes.cls.cpu().numpy()

            for box, score, track_id, cls_id in zip(xyxy, confs, ids, clss):
                if score < conf_thres:
                    continue

                # Skip person class detections (COCO class 0)
                if int(cls_id) == 0:
                    continue

                x1, y1, x2, y2 = box.astype(int)
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2

                # Reject detections outside ROI
                if not (y_min <= cy <= y_max):
                    continue

                # Check if detection overlaps with any hand region
                if filter_hands and hand_regions:
                    overlaps_hand = False
                    for hx1, hy1, hx2, hy2 in hand_regions:
                        # Check if bounding boxes overlap
                        if not (x2 < hx1 or x1 > hx2 or y2 < hy1 or y1 > hy2):
                            # Calculate intersection area
                            ix1 = max(x1, hx1)
                            iy1 = max(y1, hy1)
                            ix2 = min(x2, hx2)
                            iy2 = min(y2, hy2)

                            intersection_area = (ix2 - ix1) * (iy2 - iy1)
                            detection_area = (x2 - x1) * (y2 - y1)

                            # If more than 30% of detection overlaps with hand, skip it
                            if intersection_area / detection_area > 0.3:
                                overlaps_hand = True
                                break

                    if overlaps_hand:
                        # Draw rejected detection in red
                        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 1)
                        cv2.putText(frame, "HAND FILTERED", (x1, y1-5),
                                  cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)
                        continue

                # Crop the detection
                crop = frame[max(0, y1):min(h, y2), max(0, x1):min(w, x2)]

                if crop.size > 0:
                    # Default values
                    class_name = "UNKNOWN"
                    box_color = (0, 255, 0)
                    confidence = 0.0

                    # Classify the crop using custom model
                    classify_results = classification_model(crop, verbose=False, conf=classify_conf_thres)

                    if len(classify_results) > 0:
                        classify_result = classify_results[0]

                        # Check if it's a classification model
                        if hasattr(classify_result, 'probs') and classify_result.probs is not None:
                            probs = classify_result.probs.data.cpu().numpy()
                            predicted_class = int(probs.argmax())
                            confidence = float(probs.max())

                            class_name = "METAL" if predicted_class == 0 else "PLASTIC"
                            box_color = (255, 0, 0) if predicted_class == 0 else (0, 255, 255)

                            # Store classification
                            all_classifications.append(predicted_class)
                            if track_id != -1:
                                if track_id not in track_classifications:
                                    track_classifications[track_id] = []
                                track_classifications[track_id].append(predicted_class)

                        # Check if it's a detection model
                        elif hasattr(classify_result, 'boxes') and classify_result.boxes is not None:
                            classify_boxes = classify_result.boxes

                            if len(classify_boxes) > 0:
                                # Take highest confidence detection
                                best_idx = classify_boxes.conf.argmax()
                                predicted_class = int(classify_boxes.cls[best_idx].cpu().numpy())
                                confidence = float(classify_boxes.conf[best_idx].cpu().numpy())

                                class_name = "METAL" if predicted_class == 0 else "PLASTIC"
                                box_color = (255, 0, 0) if predicted_class == 0 else (0, 255, 255)

                                # Store classification
                                all_classifications.append(predicted_class)
                                if track_id != -1:
                                    if track_id not in track_classifications:
                                        track_classifications[track_id] = []
                                    track_classifications[track_id].append(predicted_class)
                            else:
                                class_name = "NO_DETECTION"

                    # Draw bounding box and label
                    label = f"ID:{int(track_id)} {class_name} {confidence:.2f}"
                    cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)
                    cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)
                    cv2.putText(
                        frame,
                        label,
                        (x1, max(0, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.5,
                        box_color,
                        2,
                        cv2.LINE_AA,
                    )

                    # Save crop with classification
                    if save_crops_dir is not None and track_id != -1 and class_name != "UNKNOWN":
                        crop_path = os.path.join(
                            save_crops_dir,
                            f"frame{frame_idx:05d}_id{int(track_id)}_{class_name}_{confidence:.3f}.jpg",
                        )
                        cv2.imwrite(crop_path, crop)

        # Add overall statistics overlay to video
        if all_classifications:
            class_counts = Counter(all_classifications)
            metal_count = class_counts.get(0, 0)
            plastic_count = class_counts.get(1, 0)
            total = len(all_classifications)

            # Determine current majority
            current_majority = "METAL" if metal_count > plastic_count else "PLASTIC"
            majority_color = (255, 0, 0) if metal_count > plastic_count else (0, 255, 255)

            # Create semi-transparent overlay for statistics
            overlay = frame.copy()
            cv2.rectangle(overlay, (10, 10), (400, 120), (0, 0, 0), -1)
            cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

            # Add text
            cv2.putText(frame, f"CURRENT VERDICT: {current_majority}", (20, 35),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, majority_color, 2, cv2.LINE_AA)
            cv2.putText(frame, f"Metal: {metal_count} ({metal_count/total*100:.1f}%)", (20, 65),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f"Plastic: {plastic_count} ({plastic_count/total*100:.1f}%)", (20, 95),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2, cv2.LINE_AA)

        writer.write(frame)
        frame_idx += 1

        if frame_idx % 30 == 0:
            print(f"Processed {frame_idx} frames...")

    writer.release()

    # Calculate final results
    print("\n" + "="*60)
    print("📊 CLASSIFICATION RESULTS")
    print("="*60)

    final_class_name = None
    metal_count = 0
    plastic_count = 0
    total = 0

    if all_classifications:
        class_counts = Counter(all_classifications)
        total = len(all_classifications)
        metal_count = class_counts.get(0, 0)
        plastic_count = class_counts.get(1, 0)

        metal_percent = (metal_count / total) * 100
        plastic_percent = (plastic_count / total) * 100

        final_class = 0 if metal_count > plastic_count else 1
        final_class_name = "METAL" if final_class == 0 else "PLASTIC"

        print(f"\n🎯 FINAL VERDICT: {final_class_name}")
        print(f"\nTotal classifications: {total}")
        print(f"  • METAL:   {metal_count:4d} ({metal_percent:5.1f}%)")
        print(f"  • PLASTIC: {plastic_count:4d} ({plastic_percent:5.1f}%)")

        # Per-track results
        print(f"\n📦 Per-Track Classifications:")
        for track_id, classifications in sorted(track_classifications.items()):
            track_counts = Counter(classifications)
            track_total = len(classifications)
            track_metal = track_counts.get(0, 0)
            track_plastic = track_counts.get(1, 0)
            track_final = 0 if track_metal > track_plastic else 1
            track_final_name = "METAL" if track_final == 0 else "PLASTIC"

            print(f"  Track ID {int(track_id):3d}: {track_final_name:7s} "
                  f"(M:{track_metal:3d}, P:{track_plastic:3d}, Total:{track_total:3d})")
    else:
        print("\n⚠️  No classifications made!")

    print("\n" + "="*60)
    print(f"✅ Saved tracked video to: {output_video_path}")
    if save_crops_dir:
        print(f"✅ Saved crops to: {save_crops_dir}")
    print("="*60)

    return {
        'final_class': final_class_name,
        'metal_count': metal_count,
        'plastic_count': plastic_count,
        'total_count': total,
        'track_results': track_classifications
    }


# --------------------------------------------------------------------
# ---------------------- CONFIGURATION -------------------------------
# --------------------------------------------------------------------

# Path to ORIGINAL YOLOv8 model for tracking (general object detection)
tracking_weights = "/content/yolov8m.pt"

# Path to YOUR CUSTOM trained model for classification (metal=0, plastic=1)
classification_weights = "/content/drive/MyDrive/Garbagedata/Garbagedata_Unzipped_Files/Ishita_Model/Garbagedata/Models_V3m_Runs/Model_V3m_Run1/weights/best.pt"

# Video and output paths
video_path = "/content/drive/MyDrive/Garbagedata/Plainbackground_videos/plastic/IMG_4450.mov"
output_video_path = "/content/drive/MyDrive/Garbagedata/Plainbackground_videos/plastic/plastic_outputs_trackclass/IMG_4450.mp4"
save_crops_dir = "/content/drive/MyDrive/Garbagedata/Plainbackground_videos/plastic/plastic_outputs/crops_classified/"

# ROI + confidence
roi_top = 0.10
roi_bottom = 0.70
conf = 0.4
classify_conf = 0.1  # Lower threshold for classification to catch more detections

# Run the script
results = track_and_classify_realtime(
    tracking_weights_path=tracking_weights,
    classification_weights_path=classification_weights,
    video_path=video_path,
    output_video_path=output_video_path,
    roi_top=roi_top,
    roi_bottom=roi_bottom,
    conf_thres=conf,
    classify_conf_thres=classify_conf,
    save_crops_dir=save_crops_dir,
    filter_hands=True,
    hand_conf_threshold=0.3,
)

print(f"\n🏆 FINAL RESULT: {results['final_class']}")
print(f"   Metal: {results['metal_count']} | Plastic: {results['plastic_count']}")

Loading tracking model...
Loading classification model...
✓ Classification model loaded (Task: detect)
Loading pose model for hand detection...

🎬 Starting tracking and classification...
✋ Hand filtering enabled - detections overlapping with hands will be excluded
Processed 30 frames...
Processed 60 frames...
Processed 90 frames...
Processed 120 frames...

📊 CLASSIFICATION RESULTS

🎯 FINAL VERDICT: PLASTIC

Total classifications: 12
  • METAL:      0 (  0.0%)
  • PLASTIC:   12 (100.0%)

📦 Per-Track Classifications:
  Track ID   1: PLASTIC (M:  0, P: 12, Total: 12)

✅ Saved tracked video to: /content/drive/MyDrive/Garbagedata/Plainbackground_videos/plastic/plastic_outputs_trackclass/IMG_4450.mp4
✅ Saved crops to: /content/drive/MyDrive/Garbagedata/Plainbackground_videos/plastic/plastic_outputs/crops_classified/

🏆 FINAL RESULT: PLASTIC
   Metal: 0 | Plastic: 12
